# Tool 8: Manually reject flagged epochs, Curry twin (Phase 2b)

*Curry twin: functionally identical to the EDF tool 8. Tool 8 reads only tool-7 `*_all-epo.fif` +
its optional companions, which the Curry tool 7 writes in the same MNE/TSV format, so no analysis
code differs: only the shared-library import path. The high-density support (32–64 channel
montages) lives in the shared `tools/qc_rejected_epochs_lib.py` and adapts to the channel count.
Generated by `tools_curry/_make_tool8_curry.py`. Do not hand-edit: edit the EDF notebook and
re-run the generator.*

Inspect the epochs that **7_preprocessing** flagged, decide **keep / reject** for each, and save the
decision as two tables (`*_epoch_decision.tsv`, `*_channel_decision.tsv`) that tool 9 joins to the epochs:
no epoch is copied, so a decision can be revised at any time. Reads one participant's `*_all-epo.fif` (+ optional
`*_preprocessing_params.json`) from the selected *raw-epochs* folder: the raw `.cdt` is never reloaded (EEG-only,
whatever channels the `.fif` holds).

**Workflow:** ① Load a participant (+ optional **channel triage**) → ② Global per-stage report →
③ Navigate rejected epochs → ④ Override decisions & save. Tool-7 outputs are never modified.

*High-density montages (32–64 channels) are supported: the montage, the metric table and the 1/f
cost all adapt to the channel count, and Section 1 lets you drop globally-bad channels and switch
the epoch rule away from “any flagged channel”. On a 3–6 channel PSG montage nothing changes.*

In [ ]:
try:
    import os, sys, json, io
    from pathlib import Path
    import numpy as np
    import pandas as pd
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt

    import ipywidgets as widgets
    from IPython.display import display, HTML, clear_output
    from ipyfilechooser import FileChooser

    # Make the shared library importable whether Voila is launched from the repo root (lib in
    # tools/) or from tools_curry/ (this twin lives there; the shared lib stays in ../tools).
    _here = os.getcwd()
    _lib_candidates = [_here, os.path.join(_here, 'tools'),
                       os.path.join(os.path.dirname(_here), 'tools'),
                       os.path.join(_here, '..', 'tools')]
    for _cand in _lib_candidates:
        if os.path.isfile(os.path.join(_cand, 'qc_rejected_epochs_lib.py')):
            _cand = os.path.abspath(_cand)
            if _cand not in sys.path:
                sys.path.insert(0, _cand)
            break
    import qc_rejected_epochs_lib as L
    import participant_selection_lib as PSL   # exclusion registry (same folder as the lib)
except ImportError as e:
    print("⚠️ Import error, a package is missing:", e)
else:
    print("✅ Packages imported successfully!")

# Shared mutable state across sections.
S = {}

# Output areas (one per section).
out_load   = widgets.Output()
out_report = widgets.Output()
out_nav    = widgets.Output()
out_review = widgets.Output()

def show_fig(fig, close=True):
    # Render a matplotlib Figure as a PNG widget — reliable under Voila with the Agg backend
    # (plain display(fig) would only print the figure's text repr). Same pattern as tools 0/6.
    # close=False keeps the Figure open so it can be reused (e.g. saved into an mne.Report afterwards).
    buf = io.BytesIO()
    fig.savefig(buf, format='png', dpi=100, bbox_inches='tight')
    if close:
        plt.close(fig)
    display(widgets.Image(value=buf.getvalue(), format='png'))

if not L.HAS_SPECPARAM:
    print('WARNING: specparam not available, 1/f metrics will be blank.')


## Section 1: Load a participant

In [ ]:
# ---- Section 1: folder pickers + participant + selection (stages / methods / event types / channels) ----
fc_data = FileChooser(os.getcwd())
fc_data.show_only_dirs = True
fc_data.title = ('<b>Data folder</b> (holds <code>derivatives/</code> + <code>reports_preprocessing/</code>) '
                 '&mdash; optional: pre-points the raw folder and anchors the reports output:')

fc_raw = FileChooser(os.getcwd())
fc_raw.show_only_dirs = True
fc_raw.title = ('<b>Raw-epochs folder</b> (holds the tool-7 <code>*_all-epo.fif</code>, e.g. '
                '<code>derivatives/raw_epo</code>):')

# OPTIONAL third picker: the tool-7 reports folder, read ONLY for {file_id}_epoch_channel_rejection.tsv
# (the per-(epoch, channel) flags). It is what makes per-channel badness exact — it carries the
# per-channel 1/f flags, which tool 8 would otherwise have to refit (~8 min on a 32-channel night).
# Leave it unset and the channel view falls back to time-domain flags recomputed from the signal.
fc_reports = FileChooser(os.getcwd())
fc_reports.show_only_dirs = True
fc_reports.title = ('<b>Reports folder</b> (holds the tool-7 <code>*_epoch_channel_rejection.tsv</code>, e.g. '
                    '<code>reports_preprocessing</code>), <i>optional</i>: exact per-channel flags:')

dd_part    = widgets.Dropdown(description='Participant:', options=[], style={'description_width':'110px'},
                              layout=widgets.Layout(width='340px'))
txt_custom = widgets.Text(description='Custom stages:', value='', style={'description_width':'110px'},
                          layout=widgets.Layout(width='340px'),
                          placeholder='e.g. N4  (comma-separated, optional)')
lbl_scan   = widgets.HTML('<i>Select the raw-epochs folder (optionally set the data folder first).</i>')
lbl_done   = widgets.HTML('')     # 'already processed' badge for the selected participant (informative only)
# Participant exclusion (shared registry, source 8_manual): a night found unusable during the review
lbl_excl        = widgets.HTML('')
txt_excl_reason = widgets.Text(placeholder='reason, e.g. electrode off from 2 a.m. on every channel',
                               layout=widgets.Layout(width='460px'))
btn_excl        = widgets.Button(description='Exclude this participant', icon='ban',
                                 layout=widgets.Layout(width='220px'))
btn_load   = widgets.Button(description='Load participant', button_style='primary', icon='folder-open')

# --- Selection: which stages / methods / event types define a 'rejected' epoch (a la tool 8bis) ---
# Stages + methods are picked BEFORE load (they also skip the slow PSD/1-f work on out-of-scope stages);
# the event-type sub-boxes are data-dependent so they populate AT load (all ticked). Change a tick then
# click 'Apply selection' to recompute the rejected-epoch set (this resets any manual overrides).
_mstyle     = {'description_width':'initial'}
_cblay      = widgets.Layout(width='auto', margin='2px 12px 2px 0')
box_stages  = widgets.HBox([])
_stage_cb   = {}
_method_cb  = {m: widgets.Checkbox(value=True, description=L.METHOD_LABEL[m], indent=False,
                                   style=_mstyle, layout=_cblay) for m in L.METHOD_ORDER}
box_methods = widgets.HBox([widgets.HTML('<b>Methods:</b>')] + [_method_cb[m] for m in L.METHOD_ORDER])
box_evt     = widgets.Box([widgets.HTML('<small style="color:#888;">event types appear here after load</small>')],
                          layout=widgets.Layout(display='flex', flex_flow='row wrap', width='100%'))
_evt_cb     = {}
# Reveal the event-type box from the 'event' method checkbox (initial display derived from its value).
box_evt.layout.display = '' if _method_cb['event'].value else 'none'
def _toggle_evt(change):
    box_evt.layout.display = '' if change['new'] else 'none'
_method_cb['event'].observe(_toggle_evt, names='value')
btn_apply   = widgets.Button(description='Apply selection', icon='sync', disabled=True,
                             tooltip='Recompute the rejected-epoch set from the ticked stages / methods / '
                                     'event types / channels (resets manual overrides)')

# --- Channel triage (matters on a HIGH-DENSITY montage) -------------------------------------------
# Tool 7 does not flag epochs, it flags (epoch x channel) PAIRS. Two questions follow, in this order:
#   1. which CHANNELS do I keep? A channel flagged on most of the night is a bad electrode: drop it
#      (it then flags no epoch at all and is recorded as dropped). Same rule as tool 8bis.
#   2. which EPOCHS do I reject, among the KEPT channels? 'any flagged channel' (the classic rule), or
#      only when more than P% of them are flagged (tool 8bis's rule).
# The order matters: a dropped channel's flags no longer count towards the epoch decision.
# Why this exists: with 3 channels 'any flagged channel' is sound; on a real 32-channel Curry night it
# rejects 72.6% of the epochs, a third of the file on a SINGLE electrode out of 32 - where the right
# action is to drop that electrode, not 30 s of EEG.
# Both default to the classic behaviour (drop nothing, rule 'any'), so out of the box the decision is
# exactly tool 7's and a sparse PSG montage is completely unaffected.
DROP_PCT_SUGGESTED = 20.0        # only used to word the load-time hint (tool 8bis's default value)
_chan_cb   = {}
box_chan   = widgets.Box([widgets.HTML('<small style="color:#888;">channels appear here after load</small>')],
                         layout=widgets.Layout(display='flex', flex_flow='row wrap', width='100%',
                                               max_height='170px', overflow_y='auto',
                                               border='1px solid #ddd', padding='4px'))
ft_drop_pct = widgets.FloatText(description='Drop channels flagged in more than (%) of the epochs:',
                                value=0.0, step=5, style={'description_width':'330px'},
                                layout=widgets.Layout(width='410px'))
lbl_drop_hint = widgets.HTML(
    '<small><b>How to read it.</b> The <b>%</b> beside each channel name below is the share of the '
    'epochs where <i>that</i> channel was flagged: its badness over the whole night. This field '
    'unticks every channel above the value you type.<br>'
    '<b>0 = keep every channel</b> (the default: the decision then matches tool 7 exactly). '
    '<i>Example:</i> type 20 (the value tool 8bis uses) and a channel flagged in 25&nbsp;% of the epochs '
    'is unticked, while one flagged in 12&nbsp;% is kept.<br>'
    '<b>When to use it.</b> A single electrode that is noisy all night long would otherwise condemn '
    'every epoch it touches: dropping the electrode once costs you one channel, keeping it can cost you '
    'half the night. On a 3&ndash;6 channel PSG montage you usually keep everything (0) and reject the '
    'bad epochs instead: there is no spare channel.<br>'
    'You can re-tick any channel by hand afterwards, but <b>changing the value re-applies the rule and '
    'discards those manual corrections</b>. An unticked channel flags no epoch and is recorded as '
    'dropped in the saved decision (tool 9 still computes it, flagged, and leaves it out of the figures).</small>')
lbl_badness = widgets.HTML('')
dd_rule = widgets.Dropdown(description='Epoch rule:', value='any',
                           options=[('any flagged channel (classic)', 'any'),
                                    ('> P % of the kept channels', 'pct')],
                           style={'description_width':'80px'}, layout=widgets.Layout(width='330px'))
ft_rule_value = widgets.FloatText(description='P (%)', value=20.0, step=5,
                                  style={'description_width':'50px'}, layout=widgets.Layout(width='130px'))
# Dropdown-revealed widget: the initial display MUST follow the current value, not a hard-coded 'none'
# (an observer only fires on a *change*) - see SPEC 'Checkbox-revealed parameter widgets'.
ft_rule_value.layout.display = '' if dd_rule.value != 'any' else 'none'
def _toggle_rule(change):
    ft_rule_value.layout.display = '' if change['new'] != 'any' else 'none'
dd_rule.observe(_toggle_rule, names='value')
lbl_rule_hint = widgets.HTML(
    '<small>An epoch holds one trace per kept channel, and tool 7 flagged them one by one. This says how '
    'many of them must be bad before the <b>whole epoch</b> is thrown away.<br>'
    '&bull; <b>any flagged channel</b> (classic, the default): one bad channel is enough. Right on a '
    '3&ndash;6 channel PSG montage, with so few electrodes, a bad one really does spoil the epoch, '
    'and this reproduces tool 7 exactly.<br>'
    '&bull; <b>&gt; P&nbsp;% of the kept channels</b>: the epoch survives unless more than P&nbsp;% of the '
    'channels are flagged. <i>Example:</i> 32 channels kept and P&nbsp;=&nbsp;20 &rarr; at least 7 must be '
    'flagged. Use it on a dense montage, where "any" rejects most of the night because of one or two '
    'electrodes. (This is the rule tool 8bis applies automatically.)</small>')
box_triage = widgets.VBox([
    widgets.HTML('<b>Channels kept &amp; how an epoch gets rejected</b>: tool 7 did not judge whole '
                 'epochs: it checked <b>every channel of every epoch separately</b> and flagged the bad '
                 '<i>(epoch &times; channel)</i> pairs. Two questions follow, <b>in this order</b>: '
                 '<b>(1)</b> which electrodes do I keep for the whole night? then '
                 '<b>(2)</b> how many of the kept ones must be flagged before I throw the epoch away? '
                 'The order matters: a dropped electrode no longer counts in question (2). '
                 'Leave both at their defaults (0 and "any flagged channel") and the decision is exactly '
                 'the one tool 7 stored.'),
    widgets.HTML('<b>(1) Which electrodes do I keep?</b>'),
    ft_drop_pct, lbl_drop_hint,
    box_chan, lbl_badness,
    widgets.HTML('<b>(2) How many flagged channels reject an epoch?</b>'),
    widgets.HBox([dd_rule, ft_rule_value]), lbl_rule_hint,
])

def _build_stage_boxes():
    _stage_cb.clear()
    stages = ['W','N1','N2','N3','R'] + [s for s in L.parse_custom_field(txt_custom.value)
                                         if s not in ('W','N1','N2','N3','R')]
    for st in stages:
        _stage_cb[st] = widgets.Checkbox(value=True, description=st, indent=False,
                                         style=_mstyle, layout=widgets.Layout(width='auto', margin='2px 10px 2px 0'))
    box_stages.children = [widgets.HTML('<b>Stages:</b>')] + [_stage_cb[s] for s in stages]

_build_stage_boxes()
txt_custom.observe(lambda chg: _build_stage_boxes(), 'value')

def _selected_stages():
    return [s for s, cb in _stage_cb.items() if cb.value]
def _selected_methods():
    return [m for m in L.METHOD_ORDER if _method_cb[m].value]
def _selected_event_types():
    return [t for t, cb in _evt_cb.items() if cb.value]
def _dropped_channels():
    return [c for c, cb in _chan_cb.items() if not cb.value]
def _epoch_rule():
    return dd_rule.value, float(ft_rule_value.value)

def _populate_event_types(evt_types):
    _evt_cb.clear()
    if not evt_types:
        box_evt.children = [widgets.HTML('<small style="color:#888;">no event types in this file '
                                         '(tool-7 event flagging did not run)</small>')]
        return
    for t in evt_types:
        _evt_cb[t] = widgets.Checkbox(value=True, description=t, indent=False, style=_mstyle, layout=_cblay)
    box_evt.children = [widgets.HTML('<b>Event types:</b>')] + [_evt_cb[t] for t in evt_types]

def _populate_channels(ch_names):
    _chan_cb.clear()
    for c in ch_names:
        _chan_cb[c] = widgets.Checkbox(value=True, description=c, indent=False, style=_mstyle,
                                       layout=widgets.Layout(width='auto', margin='1px 10px 1px 0'))
    box_chan.children = list(_chan_cb.values())

def _refresh_channel_labels():
    # Per-channel badness (% of the in-scope epochs a selected method flags the channel in) shown right
    # in the checkbox label, so "why is this one unticked?" reads without a second lookup. The dict KEY
    # stays the plain channel name - only the description changes.
    bad = S.get('badness')
    for i, (c, cb) in enumerate(_chan_cb.items()):
        cb.description = c if bad is None else f'{c}: {bad[i]:.0f}%'

def _refresh_badness_label():
    # Provenance of the per-channel flags (the badness itself is in each channel's label above).
    src = S.get('flags_source', '')
    if not src:
        lbl_badness.value = ''
        return
    warn = ('' if 'reports' in src else
            ' <span style="color:#e65100;">&#9888; time-domain methods only, select the reports '
            'folder for exact per-channel 1/f flags</span>')
    lbl_badness.value = f'<small>Channel flags: {src}{warn}</small>'

def _output_folders(folder):
    # decision tables beside the raw folder (derivatives/rejection_manual/); reports beside
    # reports_preprocessing/ (subtree relative to raw).
    data_root = Path(S['data_root'])
    try:
        subtree = Path(folder).relative_to(Path(S['raw_root']))
    except Exception:
        subtree = Path('.')
    return _decision_folder(folder), data_root / 'reports_rejection_manual' / subtree

def _decision_folder(folder):
    # epoch/channel decision tables beside the data (derivatives/rejection_manual/<subtree>/): tool 9
    # joins them to tool 7's *_all-epo.fif, so a decision can change without rewriting any epochs.
    raw_root = Path(S['raw_root'])
    try:
        subtree = Path(folder).relative_to(raw_root)
    except Exception:
        subtree = Path('.')
    return raw_root.parent / 'rejection_manual' / subtree

def _registry_folder():
    return Path(S['data_root']) if S.get('data_root') else None

def _refresh_exclusion_ui(*_):
    """Badge + button for the participant in the dropdown, from the exclusion registry."""
    fid = dd_part.value
    if not fid:
        lbl_excl.value = ''
        return
    reg, warn = PSL.load_registry(_registry_folder())
    if warn:
        lbl_excl.value = f'<span style="color:#e69500;">&#9888; {warn}</span>'
        return
    excluded, reason = PSL.status_of(PSL.effective_status(reg), fid)
    mine = reg[(reg['source'] == '8_manual') & (reg['file_id'].map(PSL.id_key) == PSL.id_key(fid))]
    lbl_excl.value = (f'<span style="color:#c62828;"><b>&#10006; EXCLUDED participant</b>: {reason}</span> '
                      '<small>(still reviewed and saved; left out of the group results)</small>'
                      if excluded else '<span style="color:#888;">not excluded</span>')
    btn_excl.description = 'Remove my exclusion' if len(mine) else 'Exclude this participant'
    if len(mine):
        txt_excl_reason.value = mine['reason'].iloc[0]

def _on_excl(b):
    fid = dd_part.value
    folder = _registry_folder()
    if not fid or folder is None:
        lbl_excl.value = '<span style="color:#c62828;">Select the raw-epochs folder and a participant first.</span>'
        return
    try:
        if b.description.startswith('Remove'):
            PSL.remove_registry_rows(folder, '8_manual', [fid])
            txt_excl_reason.value = ''
        else:
            reason = txt_excl_reason.value.strip()
            if not reason:
                lbl_excl.value = '<span style="color:#c62828;">Give a reason for the exclusion.</span>'
                return
            PSL.write_registry_rows(folder, '8_manual', [{'file_id': fid, 'reason': reason}], replace_ids=[fid])
        _refresh_exclusion_ui()
    except Exception as e:
        lbl_excl.value = f'<span style="color:#c62828;">Not saved: {e}</span>'

btn_excl.on_click(_on_excl)
dd_part.observe(_refresh_exclusion_ui, names='value')

def _is_processed(fid, folder):
    # 'done' = both the epoch decision AND a review/decision record on disk (skip convention).
    out_folder, reports_folder = _output_folders(folder)
    fif_ok = (out_folder / f'{fid}_epoch_decision.tsv').exists()
    tsv_ok = ((reports_folder / f'{fid}_epoch_rejection_reviewed.tsv').exists()
              or (reports_folder / f'{fid}_manualreject_decision.tsv').exists())
    return fif_ok and tsv_ok

def _update_done_badge(*_):
    fid = dd_part.value
    if not fid or 'parts' not in S or fid not in S['parts']:
        lbl_done.value = ''
        return
    try:
        if _is_processed(fid, S['parts'][fid]['folder']):
            lbl_done.value = ('<span style="color:#2e7d32;">&#10003; already processed in manual '
                              '(decision + review found)</span>')
        else:
            lbl_done.value = '<span style="color:#888;">&mdash; not yet processed in manual</span>'
    except Exception:
        lbl_done.value = ''

def _derive_data_root(raw_root):
    if fc_data.selected_path:
        return Path(fc_data.selected_path)
    rr = Path(raw_root)
    for anc in [rr] + list(rr.parents):
        if anc.name == 'derivatives':
            return anc.parent
    return rr.parent

def _on_data(chooser):
    try:
        data = fc_data.selected_path
        if not data:
            return
        start = Path(data) / 'derivatives' if (Path(data) / 'derivatives').is_dir() else Path(data)
        fc_raw.reset(path=str(start))
        fc_raw.title = ('<b>Raw-epochs folder</b> (holds the tool-7 <code>*_all-epo.fif</code>, e.g. '
                        '<code>derivatives/raw_epo</code>):')
        fc_reports.reset(path=str(data))
        fc_reports.title = ('<b>Reports folder</b> (holds the tool-7 <code>*_epoch_channel_rejection.tsv</code>, '
                            'e.g. <code>reports_preprocessing</code>), <i>optional</i>: '
                            'exact per-channel flags:')
    except Exception as e:
        lbl_scan.value = f'<span style="color:#c62828">Data folder error: {e}</span>'

def _on_raw(chooser):
    try:
        raw = fc_raw.selected_path
        if not raw:
            return
        raw_root = Path(raw)
        parts = L.find_participants(raw_root)
        S['parts'] = {p['file_id']: p for p in parts}
        S['raw_root'] = raw_root
        S['data_root'] = _derive_data_root(raw_root)
        dd_part.options = [p['file_id'] for p in parts]
        cs = L.load_custom_stages(S['data_root'])
        txt_custom.value = ','.join(cs)
        _build_stage_boxes()
        n_done = sum(1 for p in parts if _is_processed(p['file_id'], p['folder']))
        colour = '#2e7d32' if parts else '#c62828'
        lbl_scan.value = (f'<span style="color:{colour}">{len(parts)} participant(s) in the raw folder '
                          f'&mdash; {n_done} already processed in manual.</span>'
                          f'<br><small>decisions &rarr; {raw_root.parent / "rejection_manual"}'
                          f'<br>reports &rarr; {S["data_root"] / "reports_rejection_manual"}</small>')
        _update_done_badge()
    except Exception as e:
        lbl_scan.value = f'<span style="color:#c62828">Scan error: {e}</span>'

fc_data.register_callback(_on_data)
fc_raw.register_callback(_on_raw)
dd_part.observe(_update_done_badge, 'value')

def _find_channel_flags_tsv(fid):
    # Locate {fid}_epoch_channel_rejection.tsv WITHIN the chosen reports folder only (never a recursive
    # scan of derivatives/, so versioned tool-7 runs never mix). Returns the folder or None.
    root = fc_reports.selected_path
    if not root:
        return None
    try:
        for f in Path(root).rglob(f'{fid}_epoch_channel_rejection.tsv'):
            return f.parent
    except Exception:
        pass
    return None

def _load_channel_flags(fid, P, thr):
    # Per-(epoch, channel) flags: exact from the tool-7 TSV when the reports folder is set, else
    # recomputed from the signal (time-domain methods only — no per-channel 1/f).
    n = len(P['stages'])
    folder = _find_channel_flags_tsv(fid)
    if folder is not None:
        pf = L.load_channel_flags(folder, fid, P['ch_names'], n)
        if pf:
            return pf, f'reports TSV ({folder.name})'
    return L.recompute_channel_flags(P, thr), 'recomputed from signal (time-domain only)'

def _recompute_reject(reset_overrides=True):
    # Recompute the rejected-epoch set from the ticked methods / stages / event types / channels.
    P = S['P']
    methods_sel = [m for m in _selected_methods() if ('flag_' + m in P['meta'].columns) or m == 'event']
    stages_sel  = _selected_stages()
    evt_sel     = _selected_event_types()
    dropped     = _dropped_channels()
    rule, rule_value = _epoch_rule()
    # The channel layer is engaged ONLY when the user actually uses it. Otherwise pair_flags stays None
    # and the decision is exactly tool 7's stored flags — important because the recomputed fallback
    # flags depend on the thresholds, which differ from the tool-7 run when no params JSON is present.
    use_channels = bool(dropped) or rule != 'any'
    base, meth, insc = L.recompute_reject(
        P['meta'], methods_sel, stages_sel, evt_sel,
        pair_flags=S.get('pair_flags') if use_channels else None,
        ch_names=P['ch_names'], dropped_channels=dropped,
        epoch_rule=rule, epoch_rule_value=rule_value)
    S['base_reject'] = base; S['reject_method'] = meth; S['in_scope'] = insc
    S['methods_sel'] = methods_sel; S['stages_sel'] = stages_sel; S['event_types_sel'] = evt_sel
    S['dropped_channels'] = dropped; S['epoch_rule'] = rule; S['epoch_rule_value'] = rule_value
    if S.get('pair_flags'):
        S['badness'], _ = L.channel_badness(S['pair_flags'], methods_sel, insc, len(P['ch_names']))
        _refresh_badness_label()
        _refresh_channel_labels()
    if reset_overrides or 'final_reject' not in S:
        S['final_reject'] = base.copy(); S['overridden'] = set()
    S['metrics'] = None   # selection changed -> recompute metrics on the next Build / Run

def _print_decision(title):
    # The SAME block after "Load participant" and after "Apply selection" — the two used to be a
    # multi-line dump vs one long sentence, so the reader could not compare a re-decision with the
    # previous one. The rejected share is the number the whole tool turns on, so it gets its own line.
    P = S['P']
    nins = int(S['in_scope'].sum()); nrej = int(S['base_reject'].sum())
    pct = 100.0 * nrej / max(nins, 1)
    n_ch = len(P['ch_names']); dropped = list(S.get('dropped_channels', ()))
    rule = S.get('epoch_rule', 'any')
    rule_txt = ('any flagged channel' if rule == 'any'
                else f'> {S.get("epoch_rule_value", 20.0):g}% of the kept channels')
    evt = S.get('event_types_sel') or []
    print(title)
    print(f'   ==>  REJECTED  {nrej} / {nins} in-scope epochs  =  {pct:.1f} %')
    print(f'        stages   : {", ".join(S["stages_sel"]) or "(none)"}')
    print(f'        methods  : {", ".join(S["methods_sel"]) or "(none)"}')
    print(f'        events   : {", ".join(evt) if evt else "(none)"}')
    print(f'        channels : {n_ch - len(dropped)}/{n_ch} kept'
          + (f'  (dropped: {", ".join(dropped)})' if dropped else '')
          + f'   |   epoch rule: {rule_txt}')

def _apply_drop_rule(*_):
    # Channel-first rule applied straight to the checkboxes: re-tick everything, then untick the
    # channels whose badness exceeds the threshold. 0 = keep every channel - the test is a strict '>',
    # so 0 would otherwise untick every channel flagged even once.
    try:
        for cb in _chan_cb.values():
            cb.value = True
        pct = float(ft_drop_pct.value)
        bad = S.get('badness')
        if pct <= 0 or bad is None:
            return
        over = set(L.channels_over_threshold(bad, S['P']['ch_names'], pct))
        for c, cb in _chan_cb.items():
            if c in over:
                cb.value = False
    except Exception as e:
        with out_load:
            print(f'Channel rule error: {e}')

ft_drop_pct.observe(_apply_drop_rule, names='value')

def _find_recording_folder(fid, fif_folder):
    # Folder holding the recording and its scored-event companions — where a manual annotation belongs
    # (beside the events it completes, so it survives any tool-7 reprocessing). Format-agnostic: the
    # folder is recognised by holding a file NAMED after the participant, whatever its extension.
    # The tool-7 subtree mirrors the data subtree, so <data_root>/<subtree> is tried first; the
    # recursive fallback skips the derivatives/ and reports_* trees. None -> caller keeps the .fif folder.
    try:
        data_root = Path(S['data_root'])
        try:
            subtree = Path(fif_folder).relative_to(Path(S['raw_root']))
        except Exception:
            subtree = Path('.')
        cand = data_root / subtree
        if cand.is_dir() and any(cand.glob(f'{fid}.*')):
            return cand
        for hit in data_root.rglob(f'{fid}.*'):
            parts = [os.path.normcase(p) for p in hit.parts]
            if 'derivatives' in parts or any(p.startswith('reports_') for p in parts):
                continue
            return hit.parent
    except Exception:
        pass
    return None

def _annot_label_options(P):
    # Annotation vocabulary: the event types this file already carries (tool-7 evt_<type> columns), then
    # the canonical labels declared in tool 4's event_remap.json, then a fixed fallback list. Canonical
    # names keep an annotation usable by tool 7's event flagging if it is merged back later.
    opts = [c[len('evt_'):] for c in P['meta'].columns if c.startswith('evt_')]
    try:
        path = Path(S['data_root']) / 'config_param' / 'event_remap.json'
        if path.exists():
            with open(path, encoding='utf-8') as f:
                remap = json.load(f)
            opts += [v for v in remap.values() if isinstance(v, str) and v]
    except Exception:
        pass                          # a missing/broken config must never block the navigator
    opts += L.MANUAL_EVENT_LABELS
    uniq = []
    for o in opts:
        if o not in uniq:
            uniq.append(o)
    return uniq

# Sections 2-4 register here how to clear themselves (one entry per section, keyed so that re-running a
# Jupyter cell replaces its entry). Loading another participant, or re-applying the selection, clears
# them all: the previous figures, navigator list and counts must never stay on screen, and a stale
# navigator list would apply a Keep/Reject toggle to the WRONG epochs of the new participant.
_RESET_HOOKS = {}

def _reset_sections(msg):
    S['dirty'] = False; S['discard_armed'] = False
    for name, hook in list(_RESET_HOOKS.items()):
        try:
            hook(msg)
        except Exception as e:
            print(f'  ⚠ could not clear {name} ({e})')

def _unsaved_guard(action):
    # Manual Keep/Reject changes live in memory until Section 4 saves them. The first click that would
    # discard them only warns; a second click goes ahead.
    n = len(S.get('overridden', ()))
    if S.get('dirty') and n and not S.get('discard_armed'):
        S['discard_armed'] = True
        print(f'⚠ Unsaved manual changes on {S.get("fid")} ({n} epoch(s) changed in Section 3): click '
              f'"{action}" again to discard them, or save them in Section 4 first.')
        return False
    S['discard_armed'] = False
    return True

def _restore_seen(fid, n_epochs):
    # An interrupted review must not lose its trail: re-read the `seen` column of a previous
    # {fid}_epoch_rejection_reviewed.tsv (guarded on the epoch count). Only `seen` is read back — the
    # decisions stay recomposed from the current selection, as they always were.
    try:
        path = S['reports_folder'] / f'{fid}_epoch_rejection_reviewed.tsv'
        if not path.exists():
            return 0
        df = pd.read_csv(path, sep='\t')
        if len(df) != n_epochs or 'seen' not in df.columns:
            return 0
        S['visited'] = {int(i) for i in np.where(df['seen'].fillna(False).astype(bool).values)[0]}
        return len(S['visited'])
    except Exception:
        return 0

def _on_load(b):
    with out_load:
        clear_output(wait=True)
        try:
            fid = dd_part.value
            if not fid:
                print('Select a participant first.'); return
            if not _unsaved_guard('Load participant'):
                return
            p = S['parts'][fid]
            P = L.load_participant(p['fif'])
            cs = L.parse_custom_field(txt_custom.value)
            thr, info = L.load_params(p['folder'], fid, custom_stages_fallback=cs)
            S.update({'P': P, 'fid': fid, 'folder': p['folder'], 'thr0': thr, 'pinfo': info,
                      'custom_stages': cs, 'freqs': None, 'psds': None, 'metrics': None})
            S['ctx']    = L.load_context_epochs(p['folder'], fid)
            S['onsets'] = L.load_event_onsets(p['folder'], fid)   # optional montage markers (Section 3)
            S['out_folder'], S['reports_folder'] = _output_folders(p['folder'])
            S['decision_folder'] = _decision_folder(p['folder'])
            _refresh_exclusion_ui()
            # Review tracking + manual annotations (Section 3): start from a clean 'seen' set, then
            # restore it from a previous review if one is on disk; load the annotations of this recording.
            S['visited'] = set()
            n_seen0 = _restore_seen(fid, len(P['stages']))
            S['annot_folder'] = _find_recording_folder(fid, p['folder'])
            S['manual_events'] = L.load_manual_events(S['annot_folder'] or p['folder'], fid)
            # Methods present -> disable/untick the absent ones; event types -> populate sub-boxes (all ticked).
            for m in L.METHOD_ORDER:
                present = ('flag_' + m in P['meta'].columns)
                _method_cb[m].disabled = not present
                if not present:
                    _method_cb[m].value = False
            evt_types = sorted(c[len('evt_'):] for c in P['meta'].columns if c.startswith('evt_'))
            _populate_event_types(evt_types)
            _populate_channels(P['ch_names'])
            S['pair_flags'], S['flags_source'] = _load_channel_flags(fid, P, thr)
            dd_rule.value = 'any'                     # every load starts from the classic decision
            _build_amp_widgets(thr, cs)
            _set_scalar_thresholds(thr)
            # Section-2 / Section-3 defaults follow the montage DENSITY: a sparse PSG montage keeps the
            # classic behaviour (fit every epoch, draw every channel, one table row per channel), a
            # dense one starts on the high-density settings. All of them stay editable.
            hd = len(P['ch_names']) > L.HD_CHANNEL_THRESHOLD
            try:
                ft_1f_n.value = L.N_1F_SUBSAMPLE_HD if hd else 0
                dd_channels.value = 'flagged' if hd else 'all'
                ft_table_rows.value = L.DEFAULT_TABLE_ROWS if hd else 0
                dd_annot_label.options = _annot_label_options(P)
            except Exception:
                pass                      # Sections 2/3 not executed yet (partially-run Jupyter)
            _recompute_reject(reset_overrides=True)
            # A drop threshold left over from the previous participant must be re-applied to THIS
            # one's channels (the observer only fires on a value change), then the decision redone.
            if float(ft_drop_pct.value) > 0:
                _apply_drop_rule()
                _recompute_reject(reset_overrides=True)
            btn_apply.disabled = False
            _reset_sections(f'Participant {fid} loaded')
            n = len(P['stages'])
            src = 'from params JSON' if info['found'] else 'DEFAULTS (no *_preprocessing_params.json found)'
            n_ch = len(P['ch_names'])
            # What the FILE holds (fixed) — what the SELECTION does with it is printed by _print_decision
            # at the end of this block, in the very same form as after "Apply selection".
            print(f'Loaded {fid}:  {n} epochs of {info["epoch_length_s"]} s')
            print(f'  channels = {P["ch_names"]}   |   sfreq = {P["sfreq"]:.0f} Hz')
            if P.get('bad_channels'):
                print(f'  deselected in tool 7 (kept in the .fif as bad, never flagged, left out here) = '
                      f'{P["bad_channels"]}')
            print(f'  flagging methods in the file = {P["methods_present"]}')
            if evt_types:
                print(f'  event types in the file = {evt_types}')
            if S['ctx'] is not None:
                print(f'  context channels = {S["ctx"]["labels"]}  (toggle in Section 3)')
            else:
                print('  context channels = none (no *_context-epo.fif companion from tool 7)')
            if S['onsets'] is not None:
                print(f'  event onsets = {len(S["onsets"])} marks (shown on the Section-3 montage)')
            # Epoch numbering — state the convention in force so a mismatch with the scoring software is
            # noticed here and not after reviewing half the night (correct it with 'Epoch n° offset').
            _n0, _e0, _c0 = L.epoch_labels(0, info['epoch_length_s'] or P['epoch_len_s'],
                                           P.get('meas_date'), 1)
            _clk = f' = {_c0}' if _c0 else ' (no start time in the .fif)'
            print(f'  epoch numbering: index 0 = #{_n0} = 00:00:00{_clk}  '
                  f'(Compumedics numbers from 1 at the recording start; adjust "Epoch n° offset" in '
                  f'Section 3 if your study differs)')
            if n_seen0:
                print(f'  review trail restored: {n_seen0} epoch(s) already seen in a previous session')
            n_annot = len(S['manual_events'])
            where = S['annot_folder'] or p['folder']
            print(f'  manual annotations = {n_annot} '
                  f'({"loaded from" if n_annot else "none yet; will be written to"} {where})')
            if S['annot_folder'] is None:
                print('  NOTE: the recording folder was not found, annotations would be written beside '
                      'the .fif instead of next to the scored-event files. Select the Data folder in '
                      'Section 1 and reload to fix this.')
            print(f'  thresholds = {src}')
            print(f'  per-channel flags = {S["flags_source"]}')
            if cs:
                print(f'  custom stages = {cs}')
            # High-density montages: warn about the memory footprint and about the 'any channel' rule,
            # whose rejection rate explodes with the channel count (see the Channels box above).
            if n_ch > L.HD_CHANNEL_THRESHOLD:
                gb = P['data_uV'].nbytes / 1e9
                print(f'  NOTE: high-density montage ({n_ch} channels), ~{2*gb:.1f} GB resident '
                      f'(signal copy {gb:.1f} GB + the epochs object). Re-run tool 7 with resampling '
                      f'to reduce it.')
                bad = S.get('badness')
                if bad is not None and (bad > DROP_PCT_SUGGESTED).any():
                    over = L.channels_over_threshold(bad, P['ch_names'], DROP_PCT_SUGGESTED)
                    print(f'  NOTE: {len(over)} channel(s) flagged on more than {DROP_PCT_SUGGESTED:g}% '
                          f'of the in-scope epochs: {over}. Every channel is KEPT by default - set '
                          f'"Drop channels flagged in more than (%) of the epochs" in Section 1 to drop '
                          f'them, and/or switch the epoch rule away from "any flagged channel", then '
                          f'click "Apply selection".')
            if _is_processed(fid, p['folder']):
                print('  NOTE: a manual decision + review already exist on disk (they will be overwritten on save).')
            print()
            _print_decision('Selection applied (all stages / methods / event types ticked = tool 7\'s '
                            'own decision):')
            print('\nProceed to Section 2 (report) or Section 3 (navigator). '
                  'Change a tick then click "Apply selection" to redecide.')
        except Exception as e:
            print(f'Load error: {e}')

def _on_apply(b):
    with out_load:
        try:
            if 'P' not in S:
                print('Load a participant first.'); return
            if not _unsaved_guard('Apply selection'):
                return
            _recompute_reject(reset_overrides=True)
            _reset_sections(f'Selection re-applied for {S["fid"]}')
            n_kept = len(S['P']['ch_names']) - len(S['dropped_channels'])
            # Blank line first: this output area is NOT cleared between applications, so successive
            # re-decisions would otherwise run into each other and be impossible to compare.
            print()
            _print_decision('Selection re-applied, manual overrides reset, re-run Section 2 / Section 3:')
            if n_kept == 0:
                print('  ⚠ Every channel is unticked: nothing can be flagged and no channel would be '
                      'kept. Tick at least one channel.')
        except Exception as e:
            print(f'Apply error: {e}')

btn_load.on_click(_on_load)
btn_apply.on_click(_on_apply)

display(widgets.VBox([
    fc_data, fc_raw, fc_reports,
    widgets.HBox([dd_part, txt_custom]), lbl_done,
    widgets.HBox([btn_excl, txt_excl_reason]), lbl_excl,
    box_stages, box_methods, box_evt,
    box_triage,
    widgets.HBox([btn_load, btn_apply]),
    lbl_scan, out_load,
]))


## Section 2: Global per-stage report

Per sleep stage: **PSD overlay** (rejected epochs coloured by method over clean epochs + IQR band),
**metric distributions** (clean vs rejected with threshold lines) and a **stage × method rejection
table**, plus a **channels × epochs flagging heatmap with a per-channel badness bar** (which electrode
carries the night's flags). The thresholds below are the ones **tool 7 actually used** (from the
participant's params, or tool-7 defaults) and are **read-only** unless you tick *Override thresholds*:
they drive the reference lines and the per-channel attribution, never the flags stored in the `.fif`.
*1/f fitting runs per (epoch × channel): ~1 min on a 3-channel night, ~8 min on a 32-channel one,
hence the subsample and parallel options, both stated in the figure when they change what is shown.*

In [ ]:
# Threshold widgets (reference lines; editable). Amplitude widgets are rebuilt per participant.
box_amp   = widgets.HBox([])
_amp_w    = {}
ft_flat   = widgets.FloatText(description='Flat <', value=1.0,   style={'description_width':'70px'}, layout=widgets.Layout(width='150px'))
ft_grad   = widgets.FloatText(description='Grad >', value=100.0, style={'description_width':'70px'}, layout=widgets.Layout(width='150px'))
ft_mae    = widgets.FloatText(description='MAE >',  value=0.15,  style={'description_width':'70px'}, layout=widgets.Layout(width='150px'))
ft_r2     = widgets.FloatText(description='R² <',   value=0.95,  style={'description_width':'70px'}, layout=widgets.Layout(width='150px'))

# --- 1/f fitting cost (matters on a HIGH-DENSITY montage) ------------------------------------------
# The fit runs per (in-scope epoch x channel): ~1 min on a 3-channel night, ~8 min on a 32-channel one.
# Two levers, both reported in the figure so a reader never mistakes a subsample for the full night:
#   * Subsample  -> fit a seeded RANDOM sample of in-scope epochs. Only the metric DISTRIBUTIONS use
#                   these values; the keep/reject flags always come from the .fif, so a subsample can
#                   never change a decision. Default N is set at load: 250 above the channel threshold,
#                   0 (= every epoch) below it, so a sparse PSG montage is unaffected.
#   * Parallel   -> joblib over epochs. EXACT (same numbers), just faster; falls back to serial on error.
cb_1f_sub = widgets.Checkbox(value=True, description='Subsample 1/f epochs', indent=False,
                             style={'description_width':'initial'}, layout=widgets.Layout(width='190px'))
ft_1f_n   = widgets.IntText(description='max', value=0, style={'description_width':'35px'},
                            layout=widgets.Layout(width='120px'))
# Revealed by the checkbox: the initial display MUST be derived from its value, never hard-coded to
# 'none' — an observer only fires on a CHANGE, and this checkbox starts TICKED (SPEC: 'Checkbox-revealed
# parameter widgets'; this is exactly the bug that hid the Curry tool-6 high-pass box).
ft_1f_n.layout.display = '' if cb_1f_sub.value else 'none'
def _toggle_1f_sub(change):
    ft_1f_n.layout.display = '' if change['new'] else 'none'
cb_1f_sub.observe(_toggle_1f_sub, names='value')
cb_1f_par = widgets.Checkbox(value=True, description='Parallel fit', indent=False,
                             style={'description_width':'initial'}, layout=widgets.Layout(width='140px'))
box_1f = widgets.HBox([widgets.HTML('<b>1/f fit:</b>'), cb_1f_sub, ft_1f_n, cb_1f_par])

def _build_amp_widgets(thr, custom_stages):
    global _amp_w
    _amp_w = {}
    order = ['W', 'N1', 'N2', 'N3', 'R'] + [c for c in custom_stages if c not in ('W','N1','N2','N3','R')]
    items = []
    for st in order:
        v = float(thr['amplitude_ptp_uV'].get(st, 250.0))
        w = widgets.FloatText(description=st, value=v, style={'description_width':'40px'},
                              layout=widgets.Layout(width='120px'))
        w.observe(_refresh_thr_status, names='value')     # keep the drift warning in sync
        _amp_w[st] = w
        items.append(w)
    box_amp.children = [widgets.HTML('<b>Amplitude p-p (µV) per stage:</b>')] + items
    _apply_thr_lock()            # rebuilt per participant -> re-apply the read-only lock

def _set_scalar_thresholds(thr):
    ft_flat.value = float(thr['flat_ptp_uV'])
    ft_grad.value = float(thr['gradient_uV_per_sample'])
    ft_mae.value  = float(thr['1f_mae_max'])
    ft_r2.value   = float(thr['1f_r2_min'])

def _collect_thresholds():
    return {
        'amplitude_ptp_uV': {st: float(w.value) for st, w in _amp_w.items()},
        'flat_ptp_uV': float(ft_flat.value),
        'gradient_uV_per_sample': float(ft_grad.value),
        '1f_mae_max': float(ft_mae.value),
        '1f_r2_min': float(ft_r2.value),
    }

# --- Threshold edition (these are TOOL 7's thresholds, not tool 8's) -------------------------------
# The values above are the ones tool 7 actually used (params JSON, or tool-7 defaults when the sidecar
# is missing). They drive the reference lines, the per-channel attribution shown in Section 3 and -
# when no reports folder is selected - the per-(epoch, channel) flags recomputed from the signal, which
# do feed the decision as soon as the channel layer is engaged. Editing them must therefore be a
# deliberate act: read-only until 'Override thresholds' is ticked, with a warning as soon as a value
# drifts away from the tool-7 run.
cb_thr_override = widgets.Checkbox(value=False, description='Override thresholds (advanced)',
                                   indent=False, style={'description_width':'initial'},
                                   layout=widgets.Layout(width='270px'))
btn_thr_reset = widgets.Button(description='Reset to tool-7 values', icon='undo', disabled=True,
                               layout=widgets.Layout(width='200px'))
lbl_thr_src  = widgets.HTML('')
lbl_thr_warn = widgets.HTML('')

def _thr_widgets():
    return [ft_flat, ft_grad, ft_mae, ft_r2] + list(_amp_w.values())

def _refresh_thr_status(*_):
    # Provenance line + an amber warning as soon as a value differs from the loaded (tool-7) one.
    info = S.get('pinfo')
    if info is None:
        lbl_thr_src.value = '<small>Thresholds: load a participant first.</small>'
        lbl_thr_warn.value = ''
        return
    src = ('read from <code>*_preprocessing_params.json</code>' if info.get('found')
           else 'tool-7 DEFAULTS (no <code>*_preprocessing_params.json</code> found)')
    lbl_thr_src.value = (f'<small>Thresholds: {src}, they drive the reference lines and the '
                         f'per-channel attribution, not the stored flags.</small>')
    thr0 = S.get('thr0')
    drift = False
    try:
        if thr0 is not None:
            cur = _collect_thresholds()
            drift = (any(abs(cur[k] - float(thr0[k])) > 1e-9
                         for k in ('flat_ptp_uV', 'gradient_uV_per_sample', '1f_mae_max', '1f_r2_min'))
                     or any(abs(v - float(thr0['amplitude_ptp_uV'].get(st, 250.0))) > 1e-9
                            for st, v in cur['amplitude_ptp_uV'].items()))
    except Exception:
        drift = False
    lbl_thr_warn.value = ('<span style="color:#e65100;">&#9888; Thresholds differ from the tool-7 run '
                          '&mdash; the reference lines and the per-channel attribution no longer match '
                          'the flags stored in the .fif.</span>' if drift else '')

def _apply_thr_lock(*_):
    # Read-only unless the user ticked the override box (the values stay visible and stay in use).
    for w in _thr_widgets():
        w.disabled = not cb_thr_override.value
    btn_thr_reset.disabled = not cb_thr_override.value
    _refresh_thr_status()

def _on_thr_reset(b):
    try:
        thr0 = S.get('thr0')
        if thr0 is None:
            return
        _set_scalar_thresholds(thr0)
        for st, w in _amp_w.items():
            w.value = float(thr0['amplitude_ptp_uV'].get(st, 250.0))
        _refresh_thr_status()
    except Exception as e:
        with out_report:
            print(f'Threshold reset error: {e}')

cb_thr_override.observe(_apply_thr_lock, names='value')
btn_thr_reset.on_click(_on_thr_reset)
for _w in (ft_flat, ft_grad, ft_mae, ft_r2):
    _w.observe(_refresh_thr_status, names='value')
_apply_thr_lock()

prog_report = widgets.IntProgress(description='1/f fit:', min=0, max=1, value=0,
                                  layout=widgets.Layout(width='320px'))
btn_run_report  = widgets.Button(description='Build, show & save HTML', button_style='primary', icon='chart-area')
btn_save_report = widgets.Button(description='Save HTML only (no inline)', icon='save',
                                 layout=widgets.Layout(width='230px'))

def _ensure_metrics(thr, verbose=True):
    # Welch PSDs + per-epoch metric values, computed once per selection (shared by both buttons).
    P = S['P']
    fit_fmin, fit_fmax = S['pinfo']['fit_range']
    if S.get('freqs') is None:
        if verbose:
            print('Computing Welch PSDs…')
        S['freqs'], S['psds'] = L.compute_psds(P['epochs'], fmax=fit_fmax,
                                               smoothing=S['pinfo'].get('psd_smoothing'))
    if S.get('metrics') is None:
        in_scope = S['in_scope']
        do_1f = ('1f_r2' in S['methods_sel']) or ('1f_error' in S['methods_sel'])
        sub_n = int(ft_1f_n.value) if cb_1f_sub.value else 0
        n_jobs = -1 if cb_1f_par.value else 1
        n_fit = int(in_scope.sum())
        if sub_n and n_fit > sub_n:
            n_fit = sub_n
        if verbose:
            print(f'Fitting 1/f on {n_fit} in-scope epoch(s)…' if do_1f
                  else '1/f methods not selected: skipping the 1/f fit.')
        prog_report.max = max(1, n_fit); prog_report.value = 0
        S['metrics'] = L.compute_epoch_metrics(P['data_uV'], S['freqs'], S['psds'], fmin=fit_fmin,
                                               fit_mask=in_scope, do_1f=do_1f,
                                               subsample_n=sub_n, n_jobs=n_jobs,
                                               progress=lambda d, t: setattr(prog_report, 'value', d))
    return S['metrics']

def _build_figs(thr):
    # Section-2 figures + table for the CURRENT selection, including the channels x epochs flagging
    # heatmap (per-channel overview — the decisive view on a dense montage).
    P = S['P']
    stage_order = L.ordered_present_stages(P['stages'][S['in_scope']], S['custom_stages'])
    return L.build_participant_report_figs(
        P, S['metrics'], S['freqs'], S['psds'], thr, S['custom_stages'],
        reject_flag=S['base_reject'], reject_method=S['reject_method'],
        stage_order=stage_order, methods=S['methods_sel'],
        pair_flags=S.get('pair_flags'), in_scope=S['in_scope'], badness_pct=S.get('badness'),
        dropped_channels=S.get('dropped_channels', ()), flags_source=S.get('flags_source', ''))

def _on_run_report(b):
    with out_report:
        clear_output(wait=True)
        try:
            if 'P' not in S:
                print('Load a participant first (Section 1).'); return
            thr = _collect_thresholds()
            _ensure_metrics(thr)
            figs, html = _build_figs(thr)
            clear_output(wait=True)
            display(HTML(html))
            for name, fig in figs:
                show_fig(fig, close=False)          # keep open so we can also save them below
            # Build == save: write the HTML report right away (no second click needed).
            try:
                S['reports_folder'].mkdir(parents=True, exist_ok=True)
                out = S['reports_folder'] / f'{S["fid"]}_qc2b_report.html'
                L.save_report_html(out, f'{S["fid"]}: QC of rejected epochs', figs, html)
                print(f'Saved report → {out}')
            except Exception as e:
                print(f'⚠ Report shown but HTML save failed: {e}')
            btn_save_report.disabled = False
        except Exception as e:
            print(f'Report error: {e}')

def _on_save_report(b):
    # Standalone: computes PSDs / 1-f metrics if Build was not clicked, then writes the HTML
    # without rendering anything inline (keeps the notebook light).
    with out_report:
        clear_output(wait=True)
        try:
            if 'P' not in S:
                print('Load a participant first (Section 1).'); return
            thr = _collect_thresholds()
            _ensure_metrics(thr)
            figs, html = _build_figs(thr)
            S['reports_folder'].mkdir(parents=True, exist_ok=True)
            out = S['reports_folder'] / f'{S["fid"]}_qc2b_report.html'
            L.save_report_html(out, f'{S["fid"]}: QC of rejected epochs', figs, html)
            print(f'Saved report → {out}')
        except Exception as e:
            print(f'Save error: {e}')

btn_run_report.on_click(_on_run_report)
btn_save_report.on_click(_on_save_report)

display(widgets.VBox([
    widgets.HBox([cb_thr_override, btn_thr_reset]),
    lbl_thr_src,
    box_amp,
    widgets.HBox([ft_flat, ft_grad, ft_mae, ft_r2]),
    lbl_thr_warn,
    box_1f,
    widgets.HBox([btn_run_report, prog_report, btn_save_report]),
    out_report,
]))

def _reset_report(msg):
    with out_report:
        clear_output()
        print(f'{msg}: click "Build, show & save HTML" to build this report.')

_RESET_HOOKS['Section 2'] = _reset_report


## Section 3: Per-epoch navigator

Walk through the rejected epochs one by one (or, with **Show**, the kept ones, every in-scope epoch, or
the out-of-scope ones in view only: see the line under the dropdowns for what each choice changes). **Montage** (± context) with the flagging cause highlighted
per channel, plus a **detail panel** (PSD + aperiodic fit beside the per-channel metric table, topomaps
on a dense montage). Use the **keep / reject** toggle to override the decision: overrides feed
Section 4.

**Where am I?** Each epoch is named by its internal index *and* by the number the scoring software shows
(`Epoch 123 (#124)`), with the elapsed and wall-clock times, so an epoch can be found back in
Compumedics. Tool 7 epochs the recording from its very start and Profusion numbers from 1, hence the
default **Epoch n° offset** of +1. Change it if your study numbers epochs differently.

**What did I already do?** A coloured banner states the decision and shouts when *you* changed it. The
strip under it shows every epoch of the current filter (pale = not yet seen, black cap = decision
changed, outline = where you are) next to a `reviewed n/N` counter. The trail is saved with the review,
so an interrupted session resumes where it stopped.

**Annotate an epoch**: pick a label (e.g. an arousal the scorer missed), optionally an offset in seconds
inside the epoch, add a comment. It is written immediately to `{file_id}_manual_events.tsv` **beside the
recording**, next to the scored-event companions (which are never modified), and drawn as a dashed line on
the montage. It never changes the keep/reject decision. Tool 7 can pick these annotations up on a later
run (checkbox *Include tool-8 manual annotations*, off by default).

On a dense montage, **Channels** restricts the montage to the flagged electrodes (the title always
states how many of how many are drawn), **Table rows** keeps the metric table readable, and the detail
panel gains **topomaps** of peak-to-peak and 1/f MAE when the recording carries electrode positions
(Curry does, Compumedics EDF does not): a single bad electrode reads as a hot spot, a whole-head
artefact as a global shift.

In [ ]:
# Which epochs the navigator walks. 'flagged' is the classic review (the epochs the Section-1 selection
# rejects); 'kept' / 'all in scope' let the reviewer reject an epoch tool 7 did not flag (manual_added);
# 'out of scope' shows the epochs of the unticked stages, view only (they are never analysed).
dd_show    = widgets.Dropdown(description='Show:', value='flagged',
                              options=[('flagged', 'flagged'), ('kept', 'kept'),
                                       ('all in scope', 'all'), ('out of scope (view only)', 'out')],
                              style={'description_width':'45px'}, layout=widgets.Layout(width='250px'))
dd_contains = widgets.Dropdown(description='Contains event:', options=['(any)'], value='(any)',
                               style={'description_width':'105px'}, layout=widgets.Layout(width='270px'))
lbl_show_help = widgets.HTML(
    '<small><b>Show</b>: <b>flagged</b> = the epochs your Section-1 selection rejects (the classic review). '
    '<b>kept</b> / <b>all in scope</b> also let you reject an epoch tool 7 did not flag (saved as '
    '<code>manual_added</code>). <b>out of scope</b> = the epochs of the stages unticked in Section 1, '
    '<b>view only</b>: they are never analysed. The Section-2 report shows the <i>automatic</i> flags; '
    'Section 4, the saved decision and tool 9 use your <i>final</i> decision on the in-scope epochs.</small>')
dd_stage   = widgets.Dropdown(description='Stage:', options=['(all)'], value='(all)',
                              style={'description_width':'55px'}, layout=widgets.Layout(width='170px'))
dd_method  = widgets.Dropdown(description='Method:', options=['(all)'], value='(all)',
                              style={'description_width':'60px'}, layout=widgets.Layout(width='190px'))
dd_spectro = widgets.Dropdown(description='Spectro ch:', options=[], style={'description_width':'80px'},
                              layout=widgets.Layout(width='230px'))
# --- High-density display levers -------------------------------------------------------------------
# On a 32-64 channel montage the median rejected epoch is flagged on ONE channel, so drawing all of
# them to find it is backwards; and a 32-row metric table is illegible. Both default to the classic
# 'show everything' and are switched at load when the montage turns out to be dense.
dd_channels = widgets.Dropdown(description='Channels:', value='all',
                               options=[('all', 'all'),
                                        ('flagged only', 'flagged'),
                                        ('flagged + neighbours', 'flagged+neighbours'),
                                        ('worst 16 (p-p)', 'worst')],
                               style={'description_width':'70px'}, layout=widgets.Layout(width='250px'))
ft_table_rows = widgets.IntText(description='Table rows', value=0, style={'description_width':'80px'},
                                layout=widgets.Layout(width='160px'))
sl_ctx     = widgets.IntSlider(description='± context', value=1, min=0, max=3, layout=widgets.Layout(width='230px'))
cb_context = widgets.Checkbox(value=True, description='Show EOG/EMG context', indent=False,
                              layout=widgets.Layout(width='210px'))
sl_epoch   = widgets.IntSlider(description='Index', value=0, min=0, max=0, continuous_update=False,
                               layout=widgets.Layout(width='400px'))
btn_prev   = widgets.Button(description='◀ Prev', layout=widgets.Layout(width='90px'))
btn_next   = widgets.Button(description='Next ▶', layout=widgets.Layout(width='90px'))
tgl_keep   = widgets.ToggleButtons(options=[('Keep','keep'), ('Reject','reject')], value='reject',
                                   style={'button_width':'90px'})
btn_apply_all = widgets.Button(description='Apply to all shown', icon='clone',
                               tooltip='Set the current Keep/Reject decision on every epoch in the current '
                                       'stage + method filter', layout=widgets.Layout(width='190px'))
btn_apply_cancel = widgets.Button(description='Cancel', layout=widgets.Layout(width='80px', display='none'))
lbl_applyall = widgets.HTML('')
lbl_pos    = widgets.HTML('')
lbl_status = widgets.HTML('')     # decision banner: unmissable when the decision was CHANGED
lbl_progress = widgets.HTML('')   # 'reviewed n/N' + one cell per epoch of the current filter
btn_run_nav = widgets.Button(description='Start navigator', button_style='primary', icon='play')
# --- Epoch numbering as the scoring software shows it ----------------------------------------------
# Tool 7 epochs the recording from t=0 with no crop, and Compumedics/Profusion numbers epochs from 1 at
# that same origin, so #N = index N-1. The offset stays editable because a recording re-exported from a
# longer study (or a setup numbering from elsewhere) breaks that +1, and the reviewer must be able to
# realign the display with what Compumedics shows.
ft_epoch_offset = widgets.IntText(description='Epoch n° offset', value=1,
                                  style={'description_width':'110px'},
                                  layout=widgets.Layout(width='185px'))
# --- Manual annotations ---------------------------------------------------------------------------
# A deliberately rudimentary dialog to note an event the scorer missed (a forgotten arousal, ...). It
# NEVER changes the keep/reject decision: it is written to {file_id}_manual_events.tsv beside the
# recording, next to the scored-event companions, which are themselves never modified.
dd_annot_label = widgets.Dropdown(description='Label:', options=L.MANUAL_EVENT_LABELS,
                                  style={'description_width':'45px'},
                                  layout=widgets.Layout(width='215px'))
ft_annot_offset = widgets.FloatText(description='at (s) in epoch', value=0.0, step=1.0,
                                    style={'description_width':'105px'},
                                    layout=widgets.Layout(width='190px'))
txt_annot_comment = widgets.Text(description='Comment:', placeholder='free text: the event name when the label is "other"',
                                 style={'description_width':'70px'},
                                 layout=widgets.Layout(width='380px'))
btn_annot_add = widgets.Button(description='Add annotation', icon='plus',
                               layout=widgets.Layout(width='160px'))
box_annot_list = widgets.VBox([])   # one row per annotation of the current epoch (+ its delete button)
lbl_annot_info = widgets.HTML('')
out_detail = widgets.Output()     # PSD + per-channel metric table (+ topomaps on a dense montage)

# Fixed display scales (µV peak-to-peak per channel row), clinical PSG conventions from the shared lib.
# They do NOT follow the epoch: keeping them constant is what makes waveform amplitudes (75 µV slow waves,
# EMG tone, ...) comparable from one epoch to the next. Editable here if a montage needs more/less room.
_sclay = widgets.Layout(width='130px')
_scsty = {'description_width': '46px'}
ft_sc_eeg = widgets.FloatText(description='EEG', value=L.DISPLAY_SCALE_UV['EEG'], step=25,
                              style=_scsty, layout=_sclay)
ft_sc_eog = widgets.FloatText(description='EOG', value=L.DISPLAY_SCALE_UV['EOG'], step=50,
                              style=_scsty, layout=_sclay)
ft_sc_emg = widgets.FloatText(description='EMG', value=L.DISPLAY_SCALE_UV['EMG'], step=25,
                              style=_scsty, layout=_sclay)
ft_sc_ecg = widgets.FloatText(description='ECG', value=L.DISPLAY_SCALE_UV['ECG'], step=100,
                              style=_scsty, layout=_sclay)
box_scales = widgets.HBox([widgets.HTML('<b>Scale (µV/row):</b>'),
                           ft_sc_eeg, ft_sc_eog, ft_sc_emg, ft_sc_ecg])

def _current_scales():
    return {'EEG': ft_sc_eeg.value, 'EOG': ft_sc_eog.value,
            'EMG': ft_sc_emg.value, 'ECG': ft_sc_ecg.value}

def _epoch_len_s():
    # Analysis epoch length: tool 7's sidecar value (authoritative), else what the .fif itself says.
    try:
        return float(S['pinfo'].get('epoch_length_s') or S['P']['epoch_len_s'])
    except Exception:
        return 30.0

def _labels_of(ei):
    # (number, elapsed, clock) for epoch ei — the number is the Compumedics one (see ft_epoch_offset).
    return L.epoch_labels(ei, _epoch_len_s(), S['P'].get('meas_date'), int(ft_epoch_offset.value))

def _clock_at(onset_s):
    # Wall-clock time of any instant in the recording, from the .fif's meas_date (no raw reload).
    md = S['P'].get('meas_date')
    if md is None:
        return ''
    try:
        return (md + pd.to_timedelta(float(onset_s), unit='s')).strftime('%H:%M:%S')
    except Exception:
        return ''

def _status_html(ei, first_view):
    # State of THIS epoch, impossible to miss: the decision, whether the reviewer CHANGED it, and
    # whether it is being seen for the first time. Wording and colour both carry it (never colour alone).
    if not bool(S['in_scope'][ei]):
        seen_txt = ' &bull; first view' if first_view else ''
        return (f'<div style="background:#eeeeee;border-left:6px solid #757575;padding:6px 10px;'
                f'font-size:1.05em;color:#424242;"><b>OUT OF SCOPE (stage {S["P"]["stages"][ei]} not selected '
                f'in Section 1): not analysed</b>, view only{seen_txt}</div>')
    rej = bool(S['final_reject'][ei])
    changed = ei in S['overridden']
    word = 'REJECT' if rej else 'KEEP'
    col = '#c62828' if rej else '#1b5e20'
    if changed:
        bg = '#ffcdd2' if rej else '#c8e6c9'
        txt = f'{word}: CHANGED from {"keep" if rej else "reject"}'
    else:
        bg = '#fafafa'
        txt = f'{word}: unchanged'
    seen_txt = ' &bull; first view' if first_view else ''
    return (f'<div style="background:{bg};border-left:6px solid {col};padding:6px 10px;'
            f'font-size:1.05em;color:{col};"><b>{txt}</b>{seen_txt}</div>')

def _progress_html():
    # 'reviewed n/N' + one cell per epoch of the CURRENT filter: colour = decision, pale = not yet seen,
    # black cap = changed, outline = current position. Pure HTML/CSS, so navigating stays instant (a
    # matplotlib strip would add a full figure render to every step).
    nav = S.get('nav_list', [])
    if not nav:
        return ''
    cur = _current_ei()
    cells = []
    for e in nav:
        rej = bool(S['final_reject'][e])
        seen = e in S['visited']
        col = ('#c0392b' if rej else '#2ecc71') if seen else ('#f2bcb6' if rej else '#c3e8d1')
        if not bool(S['in_scope'][e]):
            col = '#9e9e9e' if seen else '#e0e0e0'
        cap = 'border-top:3px solid #000;' if e in S['overridden'] else ''
        here = 'outline:2px solid #000;outline-offset:-2px;' if e == cur else ''
        cells.append(f'<span style="flex:1 1 0;min-width:1px;height:14px;background:{col};{cap}{here}"></span>')
    n_seen = sum(1 for e in nav if e in S['visited'])
    n_chg = sum(1 for e in nav if e in S['overridden'])
    return (f'<small>Reviewed <b>{n_seen}/{len(nav)}</b> of the shown set '
            f'({100.0 * n_seen / len(nav):.0f}%): decision changed on <b>{n_chg}</b>. '
            f'Pale = not yet seen, black cap = changed.</small>'
            f'<div style="display:flex;gap:1px;margin-top:3px;">{"".join(cells)}</div>')

def _annot_folder():
    # Where {file_id}_manual_events.tsv is written: the folder holding the recording and its scored-event
    # companions (resolved at participant load). Falls back to the .fif folder — with a ⚠ in the info
    # line — when the Data folder was not selected, so an annotation is never lost for want of a path.
    return S.get('annot_folder') or S.get('folder')

def _annot_index_of(ei):
    # Row labels of the annotations attached to epoch ei (coerced: a hand-edited TSV may hold anything).
    df = S.get('manual_events')
    if df is None or not len(df):
        return []
    idx = pd.to_numeric(df['epoch_idx'], errors='coerce')
    return list(df.index[idx == int(ei)])

def _save_annots():
    p = L.save_manual_events(_annot_folder(), S['fid'], S['manual_events'])
    n = len(S['manual_events'])
    if p is None:
        lbl_annot_info.value = ('<span style="color:#c62828">&#9888; could not write the annotation file '
                                '&mdash; check the folder permissions.</span>')
    else:
        warn = ('' if S.get('annot_folder') else
                ' <span style="color:#e65100">&#9888; recording folder not found (select the Data folder '
                'in Section 1): saved beside the .fif instead</span>')
        lbl_annot_info.value = f'<small>{n} annotation(s) &rarr; {p}{warn}</small>'

def _make_annot_delete(row_id):
    def _f(b):
        try:
            S['manual_events'] = S['manual_events'].drop(index=row_id)
            _save_annots()
            ei = _current_ei()
            _refresh_annot_list(ei)
            if ei is not None:
                _render_montage(ei, _collect_thresholds())
        except Exception as e:
            lbl_annot_info.value = f'<span style="color:#c62828">Delete error: {e}</span>'
    return _f

def _refresh_annot_list(ei):
    # The annotations of the CURRENT epoch, each with its own delete button (deleting elsewhere in the
    # night is done by navigating to that epoch — the TSV stays the full-file record).
    try:
        rows = []
        for rid in ([] if ei is None else _annot_index_of(ei)):
            a = S['manual_events'].loc[rid]
            cmt = str(a.get('comment', '') or '')
            cmt = '' if cmt.lower() == 'nan' else cmt
            txt = (f'<b>{a["type"]}</b> @ {float(a["onset_s"]):.1f} s  {a.get("clock_time", "")}'
                   + (f': {cmt}' if cmt else ''))
            btn = widgets.Button(description='', icon='trash', tooltip='Delete this annotation',
                                 layout=widgets.Layout(width='40px'))
            btn.on_click(_make_annot_delete(rid))
            rows.append(widgets.HBox([btn, widgets.HTML(txt)]))
        box_annot_list.children = tuple(rows)
    except Exception as e:
        box_annot_list.children = (widgets.HTML(f'<span style="color:#c62828">Annotation list error: '
                                                f'{e}</span>'),)

def _on_annot_add(b):
    # Written to disk on the spot: a Voila session that dies must never take an annotation with it.
    try:
        if 'P' not in S:
            lbl_annot_info.value = '<span style="color:#c62828">Load a participant first.</span>'
            return
        ei = _current_ei()
        if ei is None:
            return
        label = str(dd_annot_label.value)
        comment = txt_annot_comment.value.strip()
        if label == 'other':
            if not comment:
                lbl_annot_info.value = ('<span style="color:#e65100">Type the event name in Comment when '
                                        'the label is "other".</span>')
                return
            label = comment
        ep_len = _epoch_len_s()
        off = min(max(float(ft_annot_offset.value), 0.0), ep_len)   # keep the mark inside its own epoch
        onset = ei * ep_len + off
        num, _elapsed, _clock = _labels_of(ei)
        row = {'type': label, 'onset_s': round(onset, 3), 'duration_s': np.nan,
               'epoch_idx': int(ei), 'epoch_number': num, 'clock_time': _clock_at(onset),
               'stage': S['P']['stages'][ei], 'comment': comment, 'source': 'manual_tool7',
               'created_at': pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}
        new = pd.DataFrame([row])
        S['manual_events'] = (new if not len(S['manual_events'])
                              else pd.concat([S['manual_events'], new], ignore_index=True))
        _save_annots()
        txt_annot_comment.value = ''
        _refresh_annot_list(ei)
        _render_montage(ei, _collect_thresholds())
    except Exception as e:
        lbl_annot_info.value = f'<span style="color:#c62828">Annotation error: {e}</span>'

def _method_hit_array(m):
    # Boolean (n_ep,) mask: epochs that method m flags (for 'event', OR of the selected evt_<type>).
    P = S['P']; n = len(P['stages'])
    if m == 'event':
        ets = S.get('event_types_sel') or []
        cols = ['evt_' + t for t in ets if 'evt_' + t in P['meta'].columns]
        if cols:
            arr = np.zeros(n, dtype=bool)
            for c in cols:
                arr |= P['meta'][c].astype(bool).values
            return arr
        return (P['meta']['flag_event'].astype(bool).values if 'flag_event' in P['meta'].columns
                else np.zeros(n, dtype=bool))
    return (P['meta']['flag_' + m].astype(bool).values if ('flag_' + m in P['meta'].columns)
            else np.zeros(n, dtype=bool))

def _build_method_arrays():
    S['method_arrays'] = {m: _method_hit_array(m) for m in S['methods_sel']}
    # Per-event-type masks so the Method dropdown can offer 'event: <type>' and Apply-to-all can act on a
    # single event category (e.g. Keep every epoch flagged by hypopneas once they look clean on the EEG).
    P = S['P']
    S['event_type_arrays'] = {t: P['meta']['evt_' + t].astype(bool).values
                              for t in (S.get('event_types_sel') or [])
                              if 'evt_' + t in P['meta'].columns}

def _show_mask():
    # The epochs of the current 'Show' mode, built from the AUTOMATIC decision (base_reject), never from
    # the final one, so the list does not shift while the reviewer toggles Keep/Reject.
    insc = np.asarray(S['in_scope'], dtype=bool); base = np.asarray(S['base_reject'], dtype=bool)
    mode = dd_show.value
    if mode == 'kept':
        return insc & ~base
    if mode == 'all':
        return insc
    if mode == 'out':
        return ~insc
    return base

def _tool7_flag(m):
    # Tool 7's own flag for a method, whether or not it is ticked in Section 1 (None if absent).
    meta = S['P']['meta']
    return meta['flag_' + m].astype(bool).values if 'flag_' + m in meta.columns else None

def _method_filter_array(m):
    if m == 'event (any)':
        return S.get('method_arrays', {}).get('event')
    if m.startswith('event: '):
        return S.get('event_type_arrays', {}).get(m[len('event: '):])
    if m.startswith('not counted: '):
        return _tool7_flag(m[len('not counted: '):])
    return S.get('method_arrays', {}).get(m)

def _epoch_event_arrays():
    # Event types whose ONSET falls inside each epoch: tool 7's onset sidecar (every mapped event, not
    # only the types used for flagging) + the manual annotations. Lets the reviewer pick, e.g., the kept
    # epochs that contain an arousal.
    n = len(S['P']['stages']); ep_len = _epoch_len_s()
    out = {}
    for df in (S.get('onsets'), S.get('manual_events')):
        if df is None or not len(df):
            continue
        onsets = pd.to_numeric(df['onset_s'], errors='coerce').values
        for t, o in zip(df['type'].astype(str).values, onsets):
            if not np.isfinite(o):
                continue
            ei = int(o // ep_len)
            if 0 <= ei < n:
                out.setdefault(t, np.zeros(n, dtype=bool))[ei] = True
    return out

def _nav_list():
    P = S['P']; rej = list(np.where(_show_mask())[0])
    st = dd_stage.value; m = dd_method.value; ev = dd_contains.value
    if st != '(all)':
        rej = [ei for ei in rej if P['stages'][ei] == st]
    if m != '(all)':
        arr = _method_filter_array(m)
        if arr is not None:
            rej = [ei for ei in rej if arr[ei]]
    if ev != '(any)':
        arr = S.get('epoch_event_arrays', {}).get(ev)
        if arr is not None:
            rej = [ei for ei in rej if arr[ei]]
    return rej

def _refresh_filter_options():
    P = S['P']; base = S['base_reject']; shown = _show_mask(); mode = dd_show.value
    stages = L.ordered_present_stages(P['stages'][shown], S['custom_stages'])
    # Non-event methods first, then a generic 'event (any)' and one 'event: <type>' per event category
    # that actually flagged an in-scope epoch (so a specific event type can be reviewed / bulk-kept).
    non_evt = [m for m in S['methods_sel'] if m != 'event' and (S['method_arrays'][m] & base).any()]
    evt_opts = []
    ev_arr = S['method_arrays'].get('event')
    if ev_arr is not None and (ev_arr & base).any():
        evt_opts.append('event (any)')
        evt_opts += ['event: ' + t for t in (S.get('event_types_sel') or [])
                     if t in S.get('event_type_arrays', {}) and (S['event_type_arrays'][t] & base).any()]
    if mode in ('flagged', 'all'):
        meth_opts = non_evt + evt_opts        # 'all': these filter the flagged part of the list
    elif mode == 'kept':
        # A kept epoch has no SELECTED method by definition; offer the methods unticked in Section 1
        # that tool 7 still flagged on some of them.
        meth_opts = ['not counted: ' + m for m in L.METHOD_ORDER
                     if m not in S['methods_sel'] and _tool7_flag(m) is not None
                     and (_tool7_flag(m) & shown).any()]
    else:
        meth_opts = []
    S['epoch_event_arrays'] = _epoch_event_arrays()
    ev_opts = sorted(t for t, a in S['epoch_event_arrays'].items() if (a & shown).any())
    for dd in (dd_stage, dd_method, dd_contains):
        dd.unobserve(_on_filter, 'value')
    dd_stage.options = ['(all)'] + stages; dd_stage.value = '(all)'
    dd_method.options = ['(all)'] + meth_opts; dd_method.value = '(all)'
    dd_method.disabled = not meth_opts
    dd_contains.options = ['(any)'] + ev_opts; dd_contains.value = '(any)'
    dd_contains.disabled = not ev_opts
    for dd in (dd_stage, dd_method, dd_contains):
        dd.observe(_on_filter, 'value')

def _current_ei():
    nav = S.get('nav_list', [])
    if not nav:
        return None
    return nav[min(sl_epoch.value, len(nav) - 1)]

def _methods_of(ei):
    # Names of the SELECTED methods that flagged epoch ei (resolves 'multiple').
    return [m for m in S['methods_sel'] if S.get('method_arrays', {}).get(m) is not None
            and S['method_arrays'][m][ei]]

def _montage_channels(ei, thr):
    # Channels the montage draws: the dropdown mode, minus the channels unticked in Section 1 (a
    # dropped channel is out of the decision, so it must not appear as flagging the epoch).
    # None -> every channel (kept explicit so the classic path stays byte-identical).
    P = S['P']
    dropped = set(S.get('dropped_channels', ()))
    if dd_channels.value == 'all' and not dropped:
        return None
    sel = L.select_montage_channels(P, ei, thr, mode=dd_channels.value)
    if dropped:
        sel = [i for i in sel if P['ch_names'][i] not in dropped]
    return sel or None

def _render_montage(ei, thr):
    with out_nav:
        clear_output(wait=True)
        try:
            ctx = S.get('ctx') if cb_context.value else None
            num, _elapsed, clock = _labels_of(ei)
            nlabel = f'(#{num}' + (f' — {clock}' if clock else '') + ')'
            # Manual annotations already merged into tool 7's onsets (a re-run with "Include manual
            # annotations" ticked) would otherwise be drawn twice — merge_onsets_for_display drops those.
            scored, manual = L.merge_onsets_for_display(S.get('onsets'), S.get('manual_events'))
            f1 = L.plot_epoch_montage(S['P'], ei, thr, context=int(sl_ctx.value), ctx=ctx,
                                      onsets=scored, title_method=S['reject_method'][ei] or 'none',
                                      scales=_current_scales(),
                                      show_channels=_montage_channels(ei, thr),
                                      number_label=nlabel, manual=manual)
            show_fig(f1)
        except Exception as e:
            print(f'Montage error: {e}')

def _render_detail(ei=None, thr=None):
    if ei is None:
        ei = _current_ei()
    if ei is None:
        return
    if thr is None:
        thr = _collect_thresholds()
    with out_detail:
        clear_output(wait=True)
        try:
            P = S['P']
            spi = P['ch_names'].index(dd_spectro.value) if dd_spectro.value in P['ch_names'] else 0
            # clean reference for the PSD background = in-scope epochs NOT rejected by the selection
            clean_ref = np.asarray(S['in_scope'], dtype=bool) & ~np.asarray(S['base_reject'], dtype=bool)
            f2 = L.plot_epoch_detail(P, ei, S['freqs'], S['psds'], thr, spectro_ch_idx=spi,
                                     fmin=S['pinfo']['fit_range'][0], clean_mask=clean_ref,
                                     table_rows=int(ft_table_rows.value) or None)
            show_fig(f2)
        except Exception as e:
            print(f'Detail error: {e}')

_SHOW_WORD = {'flagged': 'rejected', 'kept': 'kept', 'all': 'in-scope', 'out': 'out-of-scope'}

def _render():
    _reset_apply_all_button()
    nav = S.get('nav_list', [])
    if not nav:
        with out_nav:
            clear_output(wait=True); print(f'No {_SHOW_WORD.get(dd_show.value, "")} epochs for this filter.')
        with out_detail:
            clear_output(wait=True)
        lbl_pos.value = ''; lbl_applyall.value = ''
        lbl_status.value = ''; lbl_progress.value = ''
        box_annot_list.children = ()
        return
    ei = _current_ei()
    thr = _collect_thresholds()
    P = S['P']
    fr = 'reject' if S['final_reject'][ei] else 'keep'
    tgl_keep.unobserve(_on_toggle, 'value'); tgl_keep.value = fr; tgl_keep.observe(_on_toggle, 'value')
    # Out-of-scope epochs are never analysed: rejecting one would change nothing, so no decision there.
    in_scope_here = bool(S['in_scope'][ei])
    tgl_keep.disabled = not in_scope_here
    btn_apply_all.disabled = (dd_show.value == 'out')
    ov = ' (overridden)' if ei in S['overridden'] else ''
    # Which event type(s) flagged this epoch (tool-7 evt_<type> metadata columns).
    evt_here = [c[len('evt_'):] for c in P['meta'].columns
                if c.startswith('evt_') and bool(P['meta'].loc[ei, c])]
    evt_txt = f' — events: <b>{", ".join(evt_here)}</b>' if evt_here else ''
    # reject_method, resolving 'multiple' to the actual selected methods that flagged it.
    rm = S['reject_method'][ei]
    if rm == 'multiple':
        rm = f'multiple ({", ".join(_methods_of(ei))})'
    elif not rm:
        rm = 'none (out of scope)' if not in_scope_here else 'none (not flagged)'
    # Epoch identity: the internal 0-based index (the join key with tools 7/8bis/9), then the number the
    # scoring software shows and the times, so the epoch can be found back in Compumedics.
    num, elapsed, clock = _labels_of(ei)
    clock_txt = f' — {clock}' if clock else ''
    lbl_pos.value = (f'<b>Epoch {ei}</b> (<b>#{num}</b>) — {elapsed}{clock_txt} — '
                     f'{sl_epoch.value+1}/{len(nav)} — stage {P["stages"][ei]} '
                     f'— reject_method={rm}{evt_txt} — decision: <b>{fr}</b>{ov}')
    # 'first view' must be read BEFORE the epoch is marked as visited by this very render.
    first_view = ei not in S['visited']
    S['visited'].add(ei)
    lbl_status.value = _status_html(ei, first_view)
    lbl_progress.value = _progress_html()
    _refresh_annot_list(ei)
    # Homogeneity hint for 'Apply to all shown'.
    n_rej = int(sum(1 for e in nav if S['final_reject'][e])); n_keep = len(nav) - n_rej
    homog = ('all <b>reject</b>' if n_keep == 0 else 'all <b>keep</b>' if n_rej == 0
             else f'MIXED: {n_keep} keep / {n_rej} reject')
    if dd_show.value == 'out':
        lbl_applyall.value = (f'<small>Shown set: {len(nav)} out-of-scope epochs, view only (tick their '
                              f'stage in Section 1 and click &ldquo;Apply selection&rdquo; to review them).</small>')
    else:
        lbl_applyall.value = (f'<small>Shown set: {len(nav)} epochs, {homog}. '
                              f'&ldquo;Apply to all shown&rdquo; sets the current toggle on all of them.</small>')
    _render_montage(ei, thr)
    _render_detail(ei, thr)

def _on_run_nav(b):
    try:
        if 'P' not in S:
            with out_nav: clear_output(); print('Load a participant first (Section 1).'); return
        P = S['P']
        if S['freqs'] is None:
            S['freqs'], S['psds'] = L.compute_psds(P['epochs'], fmax=S['pinfo']['fit_range'][1],
                                                   smoothing=S['pinfo'].get('psd_smoothing'))
        dd_spectro.options = P['ch_names']; dd_spectro.value = P['ch_names'][0]
        _build_method_arrays()
        _refresh_filter_options()
        S['nav_list'] = _nav_list()
        sl_epoch.max = max(0, len(S['nav_list']) - 1); sl_epoch.value = 0
        _render()
    except Exception as e:
        with out_nav: clear_output(); print(f'Navigator error: {e}')

def _on_filter(chg):
    try:
        S['nav_list'] = _nav_list()
        sl_epoch.max = max(0, len(S['nav_list']) - 1); sl_epoch.value = 0
        _render()
    except Exception as e:
        with out_nav: print(f'Filter error: {e}')

def _step(delta):
    def _f(b):
        sl_epoch.value = int(np.clip(sl_epoch.value + delta, 0, sl_epoch.max))
    return _f

def _set_decision(ei, reject):
    if not bool(S['in_scope'][ei]):
        return                              # out of scope: never analysed, nothing to decide
    S['dirty'] = True                       # unsaved until Section 4 saves (see _unsaved_guard)
    S['final_reject'][ei] = bool(reject)
    if S['final_reject'][ei] != bool(S['base_reject'][ei]):
        S['overridden'].add(ei)
    else:
        S['overridden'].discard(ei)

def _on_toggle(chg):
    try:
        ei = _current_ei()
        if ei is None:
            return
        _set_decision(ei, chg['new'] == 'reject')
        _render()
    except Exception as e:
        with out_nav: print(f'Toggle error: {e}')

def _reset_apply_all_button():
    S['apply_all_armed'] = None
    btn_apply_all.description = 'Apply to all shown'; btn_apply_all.button_style = ''
    btn_apply_all.layout.width = '190px'
    btn_apply_cancel.layout.display = 'none'

def _on_apply_all(b):
    # Apply the current Keep/Reject toggle to EVERY epoch in the current stage + method filter.
    try:
        nav = S.get('nav_list', [])
        if not nav or dd_show.value == 'out':
            return
        reject = (tgl_keep.value == 'reject')
        # Outside the flagged list a bulk change can reach most of the night ('all in scope' + Reject =
        # every analysed epoch), so it takes a second, explicit click. Any render cancels it.
        if dd_show.value != 'flagged' and S.get('apply_all_armed') is None:
            S['apply_all_armed'] = reject
            btn_apply_all.description = f'Confirm: set {len(nav)} epochs to {"reject" if reject else "keep"}'
            btn_apply_all.button_style = 'warning'
            btn_apply_all.layout.width = '300px'
            btn_apply_cancel.layout.display = ''
            return
        if S.get('apply_all_armed') is not None:
            reject = S['apply_all_armed']
        for ei in nav:
            _set_decision(ei, reject)
        _render()
    except Exception as e:
        with out_nav: print(f'Apply-all error: {e}')

def _on_show(chg):
    try:
        if 'P' not in S or 'method_arrays' not in S:
            return                          # navigator not started yet: 'Start navigator' reads the mode
        _refresh_filter_options()
        _on_filter(None)
    except Exception as e:
        with out_nav: print(f'Show error: {e}')

def _reset_navigator(msg):
    # Called when another participant is loaded or the selection is re-applied (see _reset_sections).
    S['nav_list'] = []
    S.pop('method_arrays', None)            # the previous participant's arrays: rebuilt by Start navigator
    for dd in (dd_stage, dd_method, dd_contains):
        dd.unobserve(_on_filter, 'value')
    dd_show.unobserve(_on_show, 'value')
    dd_show.value = 'flagged'
    dd_stage.options = ['(all)']; dd_method.options = ['(all)']; dd_contains.options = ['(any)']
    dd_method.disabled = False; dd_contains.disabled = False
    for dd in (dd_stage, dd_method, dd_contains):
        dd.observe(_on_filter, 'value')
    dd_show.observe(_on_show, 'value')
    sl_epoch.max = 0; sl_epoch.value = 0
    tgl_keep.disabled = False; btn_apply_all.disabled = False
    _reset_apply_all_button()
    lbl_pos.value = ''; lbl_status.value = ''; lbl_progress.value = ''; lbl_applyall.value = ''
    box_annot_list.children = (); lbl_annot_info.value = ''
    with out_detail:
        clear_output()
    with out_nav:
        clear_output()
        print(f'{msg}: click "Start navigator".')

_RESET_HOOKS['Section 3'] = _reset_navigator

btn_run_nav.on_click(_on_run_nav)
dd_stage.observe(_on_filter, 'value')
dd_method.observe(_on_filter, 'value')
dd_contains.observe(_on_filter, 'value')
dd_show.observe(_on_show, 'value')
sl_epoch.observe(lambda chg: _render(), 'value')
sl_ctx.observe(lambda chg: (_render_montage(_current_ei(), _collect_thresholds())
                            if _current_ei() is not None else None), 'value')
cb_context.observe(lambda chg: (_render_montage(_current_ei(), _collect_thresholds())
                                if _current_ei() is not None else None), 'value')
dd_spectro.observe(lambda chg: _render_detail(), 'value')
dd_channels.observe(lambda chg: (_render_montage(_current_ei(), _collect_thresholds())
                                 if _current_ei() is not None else None), 'value')
ft_table_rows.observe(lambda chg: _render_detail(), 'value')
for _ft in (ft_sc_eeg, ft_sc_eog, ft_sc_emg, ft_sc_ecg):
    _ft.observe(lambda chg: (_render_montage(_current_ei(), _collect_thresholds())
                             if _current_ei() is not None else None), 'value')
btn_prev.on_click(_step(-1)); btn_next.on_click(_step(+1))
btn_apply_all.on_click(_on_apply_all)
btn_apply_cancel.on_click(lambda b: _reset_apply_all_button())
tgl_keep.observe(_on_toggle, 'value')
btn_annot_add.on_click(_on_annot_add)
ft_epoch_offset.observe(lambda chg: (_render() if _current_ei() is not None else None), 'value')

box_annot = widgets.VBox([
    widgets.HTML('<b>Annotate this epoch</b> <small>(e.g. an arousal the scorer missed): saved '
                 'beside the recording in <code>{file_id}_manual_events.tsv</code>; it never changes the '
                 'keep/reject decision.</small>'),
    widgets.HBox([dd_annot_label, ft_annot_offset, txt_annot_comment, btn_annot_add]),
    box_annot_list, lbl_annot_info,
], layout=widgets.Layout(border='1px solid #ddd', padding='6px', margin='6px 0 6px 0',
                         overflow='hidden'))

display(widgets.VBox([
    widgets.HBox([btn_run_nav, dd_show, dd_stage, dd_method, dd_contains]),
    lbl_show_help,
    widgets.HBox([sl_ctx, cb_context]),
    widgets.HBox([btn_prev, sl_epoch, btn_next, ft_epoch_offset]),
    widgets.HBox([dd_channels, ft_table_rows]),
    box_scales,
    widgets.HBox([widgets.HTML('<b>Decision:</b>'), tgl_keep, btn_apply_all, btn_apply_cancel]),
    lbl_applyall, lbl_pos, lbl_status, lbl_progress,
    box_annot,
    out_nav,
    out_detail,
    # The spectrogram channel selector is COMMENTED OUT with the panel it drives (see
    # qc_rejected_epochs_lib.plot_epoch_detail). The widget itself is kept alive above so the rest of
    # the code that reads dd_spectro.value keeps working — restore this row to bring the panel back.
    # widgets.HBox([widgets.HTML('<b>Spectrogram channel:</b>'), dd_spectro],
    #              layout=widgets.Layout(margin='0 0 0 695px')),
]))



## Section 4: Manual override & save

Review the final keep/reject decision (with your overrides) and save it.

**Data** → `derivatives/rejection_manual/<subtree>/`: `{file_id}_epoch_decision.tsv` (every epoch:
`in_scope` = selected stage, `rejected`, `reject_reason`, `manual_override`, `seen`) and
`{file_id}_channel_decision.tsv` (every channel: `dropped` = unticked in Section 1 or deselected in tool 7,
`drop_reason`, `badness_pct`). Tool 9 joins them to tool 7's `*_all-epo.fif`: no epoch is copied.

**Reports** → `reports_rejection_manual/<subtree>/` (beside `reports_preprocessing/`):
`{file_id}_epoch_rejection_reviewed.tsv` (per-epoch metadata + decision columns, plus `seen` and the
epoch numbering/`clock_time`),
`{file_id}_qc2b_review_log.tsv` (one row per overridden epoch),
`{file_id}_manualreject_decision.tsv` (one-row durable record: counts, %, overrides, stages/methods/event
types used, thresholds) and `{file_id}_manualreject_report.html` (visual recap of the rejected epochs).
A `global_manualreject_summary.tsv` at the root of `reports_rejection_manual/` is **rebuilt from every
decision TSV found on disk** at each save. **Tool-7 outputs are never modified.**

In [ ]:
btn_review = widgets.Button(description='Refresh review', icon='eye')
btn_save   = widgets.Button(description='Save decision + report', button_style='success', icon='save')

def _rebuild_global_summary(reports_root):
    """Global summary rebuilt by GLOBBING the per-file decision TSVs on disk (interruption-safe, same
    convention as tool 8bis) — never from an in-memory list. The participants excluded in the registry
    are flagged (participant_excluded), never dropped."""
    rows = []
    for f in sorted(Path(reports_root).rglob('*_manualreject_decision.tsv')):
        try:
            rows.append(pd.read_csv(f, sep='\t', dtype={'file_id': str}))
        except Exception:
            pass
    if not rows:
        return pd.DataFrame()
    reg, _warn = PSL.load_registry(_registry_folder())
    return PSL.add_exclusion_columns(pd.concat(rows, ignore_index=True), PSL.effective_status(reg))


def _build_decision_tables(fid, P, meta, insc, base, fr, dropped):
    """The epoch + channel decision tables (same schema as tool 8bis): every epoch with in_scope /
    rejected / why, every channel with dropped / why (tool-7 deselected channels included)."""
    over = S.get('overridden', set())
    reasons = []
    for i in range(len(fr)):
        if i in over and fr[i] and not base[i]:
            reasons.append('manual_added')
        elif i in over and base[i] and not fr[i]:
            reasons.append('manual_rescued')
        elif insc[i] and fr[i]:
            reasons.append(str(S['reject_method'][i]))
        else:
            reasons.append('')
    epoch_dec = pd.DataFrame({'file_id': fid, 'epoch_idx': meta['epoch_idx'].astype(int).values,
                              'stage': P['stages'], 'in_scope': insc, 'rejected': insc & fr,
                              'reject_reason': reasons, 'manual_override': meta['manual_override'].values,
                              'seen': meta['seen'].values})
    badness = S.get('badness')
    ch_rows = []
    for ci, ch in enumerate(P['ch_names']):
        b = float(badness[ci]) if badness is not None and ci < len(badness) else np.nan
        ch_rows.append({'file_id': fid, 'channel': ch, 'dropped': ch in dropped,
                        'drop_reason': 'unticked in tool 8' if ch in dropped else '', 'badness_pct': b})
    for ch in P.get('bad_channels', []):
        ch_rows.append({'file_id': fid, 'channel': ch, 'dropped': True, 'drop_reason': 'tool7_deselected',
                        'badness_pct': np.nan})
    return epoch_dec, pd.DataFrame(ch_rows, columns=['file_id', 'channel', 'dropped', 'drop_reason',
                                                     'badness_pct'])

def _on_review(b):
    with out_review:
        clear_output(wait=True)
        try:
            if 'P' not in S:
                print('Load a participant first (Section 1).'); return
            P = S['P']; fr = np.asarray(S['final_reject'], dtype=bool)
            insc = np.asarray(S['in_scope'], dtype=bool); base = np.asarray(S['base_reject'], dtype=bool)
            n = len(fr); n_excl = int((~insc).sum())
            keep = int((insc & ~fr).sum()); rej = int((insc & fr).sum())
            rescued = int(np.sum(insc & base & ~fr)); added = int(np.sum(insc & ~base & fr))
            print(f'Epochs: {n} | in-scope: {int(insc.sum())} | excluded (out-of-scope stages): {n_excl}')
            print(f'  final keep: {keep} | final reject: {rej} | rescued: {rescued} | newly rejected: {added} '
                  f'| overrides: {len(S["overridden"])}')
            # How much of the flagged set was actually looked at — a rejection rate says nothing about
            # the thoroughness of the review, and the two are read together.
            to_review = insc & base
            n_rev = int(sum(1 for e in S.get('visited', ()) if 0 <= e < n and to_review[e]))
            n_tr = int(to_review.sum())
            if n_tr:
                print(f'  reviewed in the navigator: {n_rev}/{n_tr} flagged epoch(s) '
                      f'({100.0 * n_rev / n_tr:.0f}%)')
            if S.get('manual_events') is not None and len(S['manual_events']):
                print(f'  manual annotations on this recording: {len(S["manual_events"])}')
            if S.get('dropped_channels'):
                print(f'  channels dropped (recorded in the decision): {S["dropped_channels"]} '
                      f'({len(P["ch_names"]) - len(S["dropped_channels"])} kept)')
            if S.get('epoch_rule', 'any') != 'any':
                print(f'  epoch rule: {S["epoch_rule"]} {S.get("epoch_rule_value")}')
            fig = L.plot_review_strip(P, fr, S['overridden'], custom_stages=S['custom_stages'],
                                      in_scope=S['in_scope'], visited=S.get('visited'))
            show_fig(fig)
        except Exception as e:
            print(f'Review error: {e}')

def _on_save(b):
    with out_review:
        try:
            if 'P' not in S:
                print('Load a participant first (Section 1).'); return
            P = S['P']; fr = np.asarray(S['final_reject'], dtype=bool)
            fid = S['fid']
            # Guard: unticking every channel would write a 0-channel file (MNE raises). It happens
            # easily on a SPARSE montage, where a 20% badness threshold can exceed every channel —
            # the same documented edge case as tool 8bis's channel-first step.
            if len(S.get('dropped_channels', ())) >= len(P['ch_names']):
                print('⚠ Every channel is unticked: nothing to save. Tick at least one channel in '
                      'Section 1 (or raise the channel-drop threshold; 0 = keep every channel), '
                      'then click "Apply selection".')
                return
            folder = S['out_folder']; reports_folder = S['reports_folder']
            reports_root = Path(S['data_root']) / 'reports_rejection_manual'
            folder.mkdir(parents=True, exist_ok=True)
            reports_folder.mkdir(parents=True, exist_ok=True)
            insc = np.asarray(S['in_scope'], dtype=bool)
            base = np.asarray(S['base_reject'], dtype=bool)
            thr = _collect_thresholds()

            # --- DATA first (durable per-item outputs), report afterwards ---
            meta = P['meta'].copy()
            meta['in_scope'] = insc
            meta['base_reject'] = base
            meta['reject_method_sel'] = S['reject_method']
            meta['manual_override'] = [i in S['overridden'] for i in range(len(fr))]
            meta['final_reject'] = fr
            dropped = [c for c in S.get('dropped_channels', ()) ]
            # Channel-triage provenance, carried in the metadata like tool 8bis's auto_reject_* columns
            # (constant per file; empty / 'any' on the classic path).
            meta['manual_dropped_channels'] = '+'.join(dropped)
            meta['manual_epoch_rule'] = S.get('epoch_rule', 'any')
            # Review trail + epoch identity as the scoring software numbers it: additive columns, so an
            # older reviewed TSV stays readable and nothing downstream has to change.
            visited = S.get('visited', set())
            meta['seen'] = [i in visited for i in range(len(fr))]
            _lab = [L.epoch_labels(i, _epoch_len_s(), P.get('meas_date'), int(ft_epoch_offset.value))
                    for i in range(len(fr))]
            meta['epoch_number'] = [t[0] for t in _lab]
            meta['onset_s'] = [round(i * _epoch_len_s(), 3) for i in range(len(fr))]
            meta['clock_time'] = [t[2] for t in _lab]
            keep_idx = np.where(insc & ~fr)[0]   # the in-scope kept epochs (what tool 9 averages)
            # the decision tables beside the data (what tool 9 joins to tool 7's epochs)
            epoch_dec, channel_dec = _build_decision_tables(fid, P, meta, insc, base, fr, dropped)
            S['decision_folder'].mkdir(parents=True, exist_ok=True)
            channel_dec.to_csv(S['decision_folder'] / f'{fid}_channel_decision.tsv', sep='\t', index=False)
            epoch_dec.to_csv(S['decision_folder'] / f'{fid}_epoch_decision.tsv', sep='\t', index=False)
            meta.to_csv(reports_folder / f'{fid}_epoch_rejection_reviewed.tsv', sep='\t', index=False)
            rows = []
            for i in sorted(S['overridden']):
                action = 'rescued' if (base[i] and not fr[i]) else \
                         ('added' if (not base[i] and fr[i]) else 'unchanged')
                rows.append({'epoch_idx': int(i), 'stage': P['stages'][i],
                             'orig_reject': bool(base[i]), 'final_reject': bool(fr[i]),
                             'action': action})
            pd.DataFrame(rows, columns=['epoch_idx','stage','orig_reject','final_reject','action']) \
                .to_csv(reports_folder / f'{fid}_qc2b_review_log.tsv', sep='\t', index=False)
            # one-row durable decision record (source of the global summary), written BEFORE the report
            decision = L.build_manual_decision_row(
                fid, P, base, fr, insc, S['overridden'], S['stages_sel'], S['methods_sel'],
                S['event_types_sel'], thr, epoch_length_s=S['pinfo']['epoch_length_s'],
                dropped_channels=dropped, epoch_rule=S.get('epoch_rule', 'any'),
                epoch_rule_value=S.get('epoch_rule_value', 20.0),
                flags_source=S.get('flags_source', ''), visited=S.get('visited'),
                epoch_number_offset=int(ft_epoch_offset.value),
                n_manual_annotations=len(S.get('manual_events', [])))
            decision.to_csv(reports_folder / f'{fid}_manualreject_decision.tsv', sep='\t', index=False)
            S['dirty'] = False; S['discard_armed'] = False     # the manual changes are now on disk
            n_kept_ch = len([c for c in P['ch_names'] if c not in dropped])
            print(f'Saved the decision: {len(keep_idx)} kept epochs × {n_kept_ch} channels '
                  f'→ {S["decision_folder"]}')
            if dropped:
                print(f'  dropped channels: {dropped}')

            # --- decision report (non-fatal) ---
            try:
                stage_order = L.ordered_present_stages(P['stages'][insc], S['custom_stages'])
                table_html = L.build_rejection_table_html(meta, S['methods_sel'], stage_order,
                                                          title='Rejection by stage × method (final decision)',
                                                          reject_flag=fr)
                html = L.manual_decision_html(fid, decision, table_html,
                                              manual_events=S.get('manual_events'))
                fig = L.plot_review_strip(P, fr, S['overridden'], custom_stages=S['custom_stages'],
                                          in_scope=insc, visited=S.get('visited'))
                L.save_report_html(reports_folder / f'{fid}_manualreject_report.html',
                                   f'{fid}: manual epoch rejection', [('Final decision', fig)], html)
                print(f'  report  → {reports_folder / (fid + "_manualreject_report.html")}')
            except Exception as rep_e:
                print(f'  ⚠ Report failed ({rep_e}): the data files above were written.')

            # --- global summary, rebuilt from every decision TSV on disk (non-fatal) ---
            try:
                summary = _rebuild_global_summary(reports_root)
                if not summary.empty:
                    summary = summary.sort_values('file_id')
                    reports_root.mkdir(parents=True, exist_ok=True)
                    summary.to_csv(reports_root / 'global_manualreject_summary.tsv', sep='\t', index=False)
                    print(f'  global summary → {reports_root / "global_manualreject_summary.tsv"} '
                          f'({len(summary)} participant(s) reviewed)')
                    cols = [c for c in ['file_id', 'n_in_scope', 'n_rejected', 'pct_rejected',
                                        'n_overrides', 'pct_seen', 'n_manual_annotations',
                                        'stages_used', 'methods_used', PSL.COL_REASON]
                            if c in summary.columns]
                    display(HTML(summary[cols].to_html(index=False)))
            except Exception as sum_e:
                print(f'  ⚠ Global summary failed ({sum_e}).')
        except Exception as e:
            print(f'Save error: {e}')

btn_review.on_click(_on_review)
btn_save.on_click(_on_save)

def _reset_review(msg):
    with out_review:
        clear_output()
        print(f'{msg}: review in Section 3, then "Refresh review" or "Save decision + report".')

_RESET_HOOKS['Section 4'] = _reset_review
display(widgets.VBox([widgets.HBox([btn_review, btn_save]), out_review]))

